# Embedding-Based Models for AI vs Human Writing Detection

This notebook extends the classical feature-based baseline by evaluating embedding-based approaches for distinguishing AI-generated and human-written academic abstracts.

Dense semantic representations generated by Word2Vec and transformer-based embeddings are combined with multiple classifiers and ensemble methods, then compared against baseline approaches using a consistent evaluation framework.

The selected models are further used to estimate the proportion of AI-generated documents in two unseen text corpora.

## Objectives
- Evaluate embedding-based representations for text classification
- Compare Word2Vec and transformer embeddings (Sentence-BERT) for AI text detection
- Assess multiple downstream classifiers and ensemble methods
- Justify model selection using accuracy, ROC-AUC, precision, recall, and F1-score
- Estimate AI-generated proportions in unseen corpora and assess uncertainty in those estimates

## Embedding Methods Overview

### Word2Vec
Word2Vec learns dense vector representations by placing words with similar contextual usage near each other in embedding space. Document-level representations in this project are derived from aggregated word embeddings.

### Transformer Embeddings (Sentence-BERT)
Sentence-BERT produces contextual embeddings that capture semantic meaning at the sentence or document level. Unlike bag-of-words methods, these embeddings preserve richer semantic relationships.

## Dataset
The dataset consists of labeled academic abstracts for binary classification, along with two unlabeled test corpora used for corpus-level estimation.

> Note: The dataset was provided as part of coursework and is included here for project demonstration purposes only.

# 0. Settings

## 0.1 Import packages

In [71]:
# Standard library
import itertools
import time

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer

## 0.2 Define functions

### `show_styled_df`

In [2]:
def show_styled_df(results_df):
    """
    Display a styled dataframe with colored performance metrics.
    Only highlight metric columns to avoid coloring hyperparameter columns.
    """
    score_cols = [
        'accuracy', 'ai_f1', 'ai_precision', 'ai_recall',
        'human_f1', 'human_precision', 'human_recall', 'macro_f1'
    ]
    
    styled_df = (
        results_df
        .style
        .background_gradient(cmap="YlOrRd", subset=score_cols)  
        .format("{:.2%}", subset=score_cols)                    
    )
    
    display(styled_df)

### `random_search_classifier`

In [ ]:
def random_search_classifier(classifier, clf_param_grid, X_train, y_train, n_iter=20, ai_label='AI', human_label='Human'):
    """Randomized-search hyperparameters for a given classifier on training data.

    Args:
        classifier: sklearn estimator, e.g. LogisticRegression().
        clf_param_grid (dict): parameter distributions for RandomizedSearchCV.
        X_train (array-like): training features.
        y_train (array-like): training labels.
        n_iter (int): number of sampled combinations.

    Returns:
        RandomizedSearchCV: fitted search (best_estimator_, best_params_, cv_results_).
    """
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    scoring = {
        'accuracy': 'accuracy',
        'ai_f1': make_scorer(f1_score, pos_label=ai_label),
        'ai_precision': make_scorer(precision_score, pos_label=ai_label),
        'ai_recall': make_scorer(recall_score, pos_label=ai_label),
        'human_f1': make_scorer(f1_score, pos_label=human_label),
        'human_precision': make_scorer(precision_score, pos_label=human_label),
        'human_recall': make_scorer(recall_score, pos_label=human_label),
        'macro_f1': 'f1_macro',
    }

    search = RandomizedSearchCV(
        estimator=classifier,
        param_distributions=clf_param_grid,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring, 
        refit='accuracy', 
        n_jobs=-1,
        random_state=42,
        verbose=2,
        return_train_score=True,
    )

    search.fit(X_train, y_train)
    return search

### `show_search_results`

In [ ]:
def show_search_results(df, sort_by='mean_test_accuracy'):
    """
    Display summarized RandomizedSearchCV results.

    Args:
        df: pd.DataFrame(search.cv_results_)
        sort_by (str): Which metric to sort results by (default: 'mean_test_accuracy').
    """
    # df = pd.DataFrame(search.cv_results_)
    if 'classifier' in df.columns:
        cols_to_show = ['classifier']
    else:
        cols_to_show = []
    cols_to_show += [
        'params', 'mean_fit_time', 'mean_score_time',
        'mean_test_accuracy', 
        'mean_test_ai_f1', 'mean_test_ai_precision', 'mean_test_ai_recall', 
        'mean_test_human_f1', 'mean_test_human_precision', 'mean_test_human_recall',
        'mean_test_macro_f1'
    ]
    df_short = df[cols_to_show].sort_values(by=sort_by, ascending=False).reset_index(drop=True)

    rename_map = {
        'mean_test_accuracy': 'accuracy',
        'mean_test_ai_f1': 'ai_f1',
        'mean_test_ai_precision': 'ai_precision',
        'mean_test_ai_recall': 'ai_recall',
        'mean_test_human_f1': 'human_f1',
        'mean_test_human_precision': 'human_precision',
        'mean_test_human_recall': 'human_recall',
        'mean_test_macro_f1': 'macro_f1'
    }
    df_short = df_short.rename(columns=rename_map)
    pd.set_option('display.max_colwidth', None)
    show_styled_df(df_short)

# 1. Load the training data
Load the training data into `train_df`. About 62% of the records are labeled as AI-generated, while 38% are labeled as human-generated.

In [ ]:
train_df = pd.read_csv('../data/raw/ai_hum_text.csv')
display(train_df['Class'].value_counts())
display(train_df['Class'].value_counts(normalize=True))

Class
AI       62701
Human    37860
Name: count, dtype: int64

Class
AI       0.623512
Human    0.376488
Name: proportion, dtype: float64

In [4]:
X = train_df['Text']
y = train_df['Class']
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=2/7, random_state=42, stratify=y_temp)

# 2. Word2Vec
Word2Vec is a method that turns words into numbers so that a computer can understand them. It learns patterns from how words appear together in sentences. For example, if “doctor” often appears near “hospital,” their numbers will be close to each other. This allows the computer to capture basic word meanings and relationships.

Rather than jointly tuning Word2Vec and each classifier (which would be computationally expensive and difficult to compare), we adopt a two-stage tuning strategy.

**1. Embedding selection (global)**

We first perform embedding selection to find the best Word2Vec configuration using a Logistic Regression. For each candidate setting, the model is trained on the tokenized training set, and sentence representations are obtained by averaging the Word2Vec embeddings of individual words through the `vectorize_texts()` function. This process converts word-level embeddings into sentence-level vectors suitable for downstream classification. The resulting features are standardized to ensure that each embedding dimension contributes equally to the model. Because Word2Vec learns a shared semantic space that captures general relationships between words, the optimal embedding found with this baseline typically transfers well to other classifiers, allowing for efficiency and fair comparison.

**2. Classifier tuning (per model)**

In the second stage, the selected Word2Vec configuration is reused as a fixed embedding, and each classifier is fine-tuned individually on the same preprocessed features to compare their effectiveness under a consistent representation.

Both stages are evaluated on the validation set using accuracy, precision, recall, and F1-score for both AI and Human labels, along with a macro-average F1-score to measure overall performance.

In [5]:
def gensim_tokenize(s: str):
    # lowercase, remove punctuation, keep tokens with reasonable length, remove STOPWORDS
    toks = simple_preprocess(s, deacc=True, min_len=2, max_len=30)
    toks = [t for t in toks if t not in STOPWORDS]
    return toks

In [11]:
def vectorize_texts(model, tokenized_texts):
    """Convert tokenized sentences to fixed-length vectors by averaging word embeddings.

    Args:
        model: Trained gensim Word2Vec model.
        tokenized_texts: List[List[str]], tokenized sentences.

    Returns:
        np.ndarray: Shape (n_samples, vector_size). Each row is a sentence vector.
    """
    dim = model.vector_size
    out = []
    for toks in tokenized_texts:
        vecs = [model.wv[t] for t in toks if t in model.wv]
        out.append(np.mean(vecs, axis=0) if vecs else np.zeros(dim))
    return np.vstack(out)
    
def compare_w2v_params(params_w2v, classifier):
    """Grid-search Word2Vec hyperparameters with a given classifier.

    Trains a Word2Vec model per hyperparameter combo, converts sentences to
    averaged embeddings, standardizes features, fits the classifier, and
    records metrics on the validation set.

    Args:
        params_w2v (dict): Grid for Word2Vec. Example::
            {
                'vector_size': [100, 200, 300],
                'min_count': [2, 5],
                'sg': [0, 1],          # 0=CBOW, 1=Skip-gram
                'epochs': [10, 20]
            }
        classifier: An sklearn-compatible estimator (e.g., LogisticRegression).

    Returns:
        pandas.DataFrame: One row per combo with params and metrics:
            ['vector_size','min_count','sg','epochs','embedding','classifier',
             'accuracy','ai_f1','ai_precision','ai_recall',
             'human_f1','human_precision','human_recall','macro_f1']
    """
    rows = []

    # Create all combinations
    all_combos = list(itertools.product(
        params_w2v['vector_size'],
        params_w2v['min_count'],
        params_w2v['sg'],
        params_w2v['epochs']
    ))
    
    # Grid search over all combinations
    for vector_size, min_count, sg, epochs in tqdm(
        all_combos,
        desc="Training Word2Vec models...",
        ncols=100
    ):
        # 1) Train Word2Vec on training set only (to avoid data leakage)
        w2v = Word2Vec(
            sentences=tokenized_train,
            vector_size=vector_size,
            min_count=min_count,
            sg=sg,
            epochs=epochs,
            seed=42
        )

        # 2) Convert sentences to average word vectors
        Xtr = vectorize_texts(w2v, tokenized_train)
        Xva = vectorize_texts(w2v, tokenized_val)

        # 3) Standardize feature vectors
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xtr_s = scaler.fit_transform(Xtr)
        Xva_s = scaler.transform(Xva)

        # 4) Train classifier
        clf = clone(classifier)
        clf.fit(Xtr_s, y_train)
        y_pred = clf.predict(Xva_s)

        # 5) Compute performance metrics
        acc      = round(accuracy_score(y_val, y_pred), 4)
        f1_ai    = round(f1_score(y_val, y_pred, pos_label='AI', zero_division=0), 4)
        p_ai     = round(precision_score(y_val, y_pred, pos_label='AI', zero_division=0), 4)
        r_ai     = round(recall_score(y_val, y_pred, pos_label='AI', zero_division=0), 4)

        f1_h     = round(f1_score(y_val, y_pred, pos_label='Human', zero_division=0), 4)
        p_h      = round(precision_score(y_val, y_pred, pos_label='Human', zero_division=0), 4)
        r_h      = round(recall_score(y_val, y_pred, pos_label='Human', zero_division=0), 4)

        f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)

        params = dict(
            vector_size=vector_size, 
            min_count=min_count,
            sg=sg, 
            epochs=epochs
        )

        row = {
            **params,
            'embedding': 'Word2Vec',
            'classifier': clf.__class__.__name__,
            'accuracy': acc,
            'ai_f1': f1_ai, 'ai_precision': p_ai, 'ai_recall': r_ai,
            'human_f1': f1_h, 'human_precision': p_h, 'human_recall': r_h,
            'macro_f1': f1_macro
        }
        
        rows.append(row)

    # Merge this run's results into the main DataFrame
    results_df = pd.DataFrame(rows)
    return results_df

## 2.1 Tokenize documents
Since Word2Vec requires tokenized inputs, the `gensim_tokenize` function is used to convert `X_train`, `X_val`, and `X_test` into tokenized words. This function converts all words to lowercase, removes punctuation marks, and filters out tokens that are too short or too long (keeps words with length between 2 and 30). In addition, words in the `STOPWORDS` list, which are common words that occur frequently but carry little semantic meaning such as *in, on, he, she,* and *within*, are removed. By doing so, the input text becomes cleaner and more suitable for training the Word2Vec model.

`tokenized_train`, `tokenized_val`, `tokenized_test` are generated for further use on Word2Vec model.

In [8]:
tokenized_train = [gensim_tokenize(s) for s in X_train]
tokenized_val   = [gensim_tokenize(s) for s in X_val]
tokenized_test  = [gensim_tokenize(s) for s in X_test]

## 2.2 Embedding selection

### 2.2.1 First attempt
The following Word2Vec hyperparameters are explored in the first attempt:
- `vector_size`: [100, 200, 300] – controls how much semantic information each word vector can capture.
- `min_count`: [2, 5] – ignores words appearing fewer than 2 or 5 times to remove noisy rare tokens.
- `sg`: [0, 1] – tests both CBOW (0) and Skip-gram (1) architectures.
- `epochs`: [10, 20] – controls how many times the model iterates through the corpus; higher values improve stability.

The combination of `vector_size=300, min_count=5, sg=1, and epochs=10` achieves the best overall performance across all evaluation metrics, reaching an accuracy of 85.74% In general, a vector_size of 300 outperforms smaller dimensions, and the Skip-gram model (sg=1) tends to yield better results than CBOW (sg=0). Based on these observations, further experiments will focus on exploring similar configurations around this setting to examine whether the performance can be further improved.

In [27]:
# First attempt
params_w2v = {
    'vector_size': [100, 200, 300],
    'min_count': [2, 5],
    'sg': [0, 1],
    'epochs': [10, 20]
}

results_df = compare_w2v_params(params_w2v=params_w2v, classifier=LogisticRegression(max_iter=1000, random_state=42))

Training Word2Vec models...:   0%|                                           | 0/24 [00:00<?, ?it/s]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:   4%|█▍                                 | 1/24 [00:10<04:05, 10.66s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:   8%|██▉                                | 2/24 [00:26<05:03, 13.81s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.

In [41]:
show_styled_df(results_df.sort_values(['accuracy'], ascending=False))

,vector_size,min_count,sg,epochs,embedding,classifier,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
22,300,5,1,10,Word2Vec,LogisticRegression,85.74%,88.72%,87.52%,89.96%,80.61%,82.57%,78.75%,84.67%
17,300,2,0,20,Word2Vec,LogisticRegression,85.44%,88.45%,87.48%,89.44%,80.29%,81.84%,78.80%,84.37%
21,300,5,0,20,Word2Vec,LogisticRegression,85.42%,88.44%,87.47%,89.43%,80.27%,81.82%,78.79%,84.36%
19,300,2,1,20,Word2Vec,LogisticRegression,85.35%,88.40%,87.26%,89.58%,80.10%,81.95%,78.34%,84.25%
18,300,2,1,10,Word2Vec,LogisticRegression,85.34%,88.41%,87.13%,89.74%,80.03%,82.12%,78.05%,84.22%
23,300,5,1,20,Word2Vec,LogisticRegression,85.25%,88.33%,87.17%,89.53%,79.96%,81.84%,78.17%,84.15%
14,200,5,1,10,Word2Vec,LogisticRegression,85.12%,88.24%,86.99%,89.52%,79.75%,81.77%,77.83%,83.99%
10,200,2,1,10,Word2Vec,LogisticRegression,85.03%,88.18%,86.89%,89.51%,79.62%,81.71%,77.63%,83.90%
11,200,2,1,20,Word2Vec,LogisticRegression,85.03%,88.19%,86.82%,89.59%,79.58%,81.80%,77.48%,83.89%
15,200,5,1,20,Word2Vec,LogisticRegression,84.89%,88.09%,86.63%,89.59%,79.35%,81.73%,77.10%,83.72%


### 2.2.2 Second attempt
Combinations of larger vector_size (300–500) and higher min_count (5 or 8) are explored to examine their effect on model performance.

The combination of `vector_size=500, min_count=8, sg=1, and epochs=10` achieves the best overall performance across all evaluation metrics, reaching an accuracy of 86.12%. This suggests that increasing both the embedding dimension and the minimum word frequency threshold can improve model performance. Therefore, additional experiments will be conducted around this configuration to further explore its stability and potential gains.

Note: Slight differences from previous results are likely caused by stochastic effects in multi-threaded Word2Vec training, as worker parallelization can introduce small variations even when the random seed is fixed.


In [42]:
# Second attempt - increase vector_size and min_count
params_w2v = {
    'vector_size': [300, 400, 500],
    'min_count': [5, 8],
    'sg': [1],
    'epochs': [10, 20]
}

results_df = compare_w2v_params(params_w2v=params_w2v, classifier=LogisticRegression(max_iter=1000, random_state=42))

Training Word2Vec models...:   0%|                                           | 0/12 [00:00<?, ?it/s]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:   8%|██▉                                | 1/12 [00:31<05:41, 31.09s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_flo

In [43]:
show_styled_df(results_df.sort_values(['accuracy'], ascending=False))

,vector_size,min_count,sg,epochs,embedding,classifier,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
10,500,8,1,10,Word2Vec,LogisticRegression,86.12%,89.02%,87.84%,90.23%,81.15%,83.06%,79.32%,85.08%
8,500,5,1,10,Word2Vec,LogisticRegression,86.11%,88.99%,87.97%,90.04%,81.19%,82.84%,79.61%,85.09%
6,400,8,1,10,Word2Vec,LogisticRegression,86.03%,88.94%,87.84%,90.06%,81.05%,82.83%,79.36%,85.00%
7,400,8,1,20,Word2Vec,LogisticRegression,85.90%,88.84%,87.69%,90.02%,80.85%,82.71%,79.08%,84.85%
11,500,8,1,20,Word2Vec,LogisticRegression,85.87%,88.80%,87.76%,89.86%,80.85%,82.52%,79.25%,84.83%
5,400,5,1,20,Word2Vec,LogisticRegression,85.78%,88.73%,87.69%,89.79%,80.73%,82.40%,79.13%,84.73%
9,500,5,1,20,Word2Vec,LogisticRegression,85.78%,88.77%,87.45%,90.13%,80.62%,82.78%,78.58%,84.70%
0,300,5,1,10,Word2Vec,LogisticRegression,85.69%,88.67%,87.50%,89.88%,80.55%,82.45%,78.74%,84.61%
4,400,5,1,10,Word2Vec,LogisticRegression,85.68%,88.65%,87.58%,89.75%,80.58%,82.30%,78.92%,84.62%
3,300,8,1,20,Word2Vec,LogisticRegression,85.60%,88.60%,87.48%,89.74%,80.45%,82.25%,78.74%,84.52%


### 2.2.3 Third attempt
Combinations of larger vector_size (500-1000) and higher min_count (8 or 10) with fixed epochs (10) are explored to examine their effect on model performance.

The combination of `vector_size=1000, min_count=10, sg=1, and epochs=10` achieves the best overall performance across most evaluation metrics, reaching an accuracy of 86.65%. Since the scores continue to improve with larger embedding dimensions and higher minimum word frequency thresholds, further experiments with even larger values could be conducted to explore potential performance gains.

In [45]:
# Thrid attempt - increase vector_size and min_count
params_w2v = {
    'vector_size': [500, 600, 700, 800, 1000],
    'min_count': [8, 10],
    'sg': [1],
    'epochs': [10]
}

results_df = compare_w2v_params(params_w2v=params_w2v, classifier=LogisticRegression(max_iter=1000, random_state=42))

Training Word2Vec models...:   0%|                                           | 0/10 [00:00<?, ?it/s]

ERROR! Session/line number was not unique in database. History logging moved to new session 112


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:  10%|███▌                               | 1/10 [00:53<08:02, 53.60s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...: 

In [46]:
show_styled_df(results_df.sort_values(['accuracy'], ascending=False))

,vector_size,min_count,sg,epochs,embedding,classifier,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
9,1000,10,1,10,Word2Vec,LogisticRegression,86.65%,89.41%,88.48%,90.36%,81.96%,83.45%,80.52%,85.69%
7,800,10,1,10,Word2Vec,LogisticRegression,86.64%,89.41%,88.46%,90.37%,81.94%,83.47%,80.47%,85.67%
8,1000,8,1,10,Word2Vec,LogisticRegression,86.62%,89.38%,88.51%,90.26%,81.94%,83.33%,80.60%,85.66%
5,700,10,1,10,Word2Vec,LogisticRegression,86.58%,89.34%,88.44%,90.26%,81.86%,83.31%,80.47%,85.60%
6,800,8,1,10,Word2Vec,LogisticRegression,86.53%,89.31%,88.33%,90.32%,81.77%,83.35%,80.24%,85.54%
4,700,8,1,10,Word2Vec,LogisticRegression,86.44%,89.25%,88.23%,90.29%,81.63%,83.27%,80.06%,85.44%
3,600,10,1,10,Word2Vec,LogisticRegression,86.35%,89.19%,88.12%,90.29%,81.50%,83.23%,79.83%,85.34%
1,500,10,1,10,Word2Vec,LogisticRegression,86.29%,89.14%,88.08%,90.23%,81.42%,83.14%,79.77%,85.28%
0,500,8,1,10,Word2Vec,LogisticRegression,86.11%,88.98%,88.00%,89.98%,81.20%,82.77%,79.69%,85.09%
2,600,8,1,10,Word2Vec,LogisticRegression,86.06%,88.96%,87.89%,90.05%,81.10%,82.82%,79.45%,85.03%


### 2.2.4 Fourth attempt
Combinations of larger vector_size (1000-2000) and higher min_count (10 or 12) are explored to examine their effect on model performance.

Compared to the previous configuration (`vector_size=1000, min_count=10, sg=1, and epochs=10`), which achieved an accuracy of 86.64%, increasing the vector size to 2000 (`vector_size=2000, min_count=10, sg=1, and epochs=10`) resulted in a slightly higher accuracy of 86.95%. However, this improvement of around 0.3% comes at the cost of approximately twice the training time, as a larger vector size generates more dimensions for each word. While a higher vector_size can capture richer semantic information and potentially enhance performance, it also leads to substantially higher computational and time costs that must be carefully considered.

In [48]:
# Fourth attempt - increase vector_size and min_count
params_w2v = {
    'vector_size': [1000, 1200, 1500, 2000],
    'min_count': [10, 12],
    'sg': [1],
    'epochs': [10]
}

results_df = compare_w2v_params(params_w2v=params_w2v, classifier=LogisticRegression(max_iter=1000, random_state=42))

Training Word2Vec models...:  12%|████▍                              | 1/8 [01:50<12:56, 110.91s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:  25%|████████▊                          | 2/8 [03:34<10:38, 106.48s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:  38%|█████████████▏                     | 3/8 [05:40<09:37, 115.45s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.

In [50]:
show_styled_df(results_df.sort_values(['accuracy'], ascending=False))

,vector_size,min_count,sg,epochs,embedding,classifier,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
6,2000,10,1,10,Word2Vec,LogisticRegression,86.95%,89.63%,88.83%,90.45%,82.41%,83.70%,81.15%,86.02%
4,1500,10,1,10,Word2Vec,LogisticRegression,86.94%,89.63%,88.78%,90.49%,82.37%,83.73%,81.06%,86.00%
5,1500,12,1,10,Word2Vec,LogisticRegression,86.88%,89.59%,88.66%,90.55%,82.27%,83.78%,80.81%,85.93%
3,1200,12,1,10,Word2Vec,LogisticRegression,86.85%,89.56%,88.67%,90.46%,82.24%,83.66%,80.86%,85.90%
7,2000,12,1,10,Word2Vec,LogisticRegression,86.79%,89.50%,88.74%,90.26%,82.20%,83.40%,81.04%,85.85%
2,1200,10,1,10,Word2Vec,LogisticRegression,86.73%,89.46%,88.59%,90.36%,82.08%,83.49%,80.72%,85.77%
0,1000,10,1,10,Word2Vec,LogisticRegression,86.72%,89.47%,88.51%,90.45%,82.04%,83.59%,80.55%,85.76%
1,1000,12,1,10,Word2Vec,LogisticRegression,86.64%,89.39%,88.56%,90.23%,81.97%,83.30%,80.69%,85.68%


### 2.2.5 Fifth attempt

Previous strong configurations are re-evaluated to verify stability and select a single embedding for downstream classifier tuning. In prior trials, `vector_size=2000` yielded only a ~0.3% improvement over `vector_size=1000` while requiring roughly twice the training time. Given the minimal performance gain and substantial computational cost, it was excluded from further experiments.

The results from this attempt are consistent with earlier findings, confirming their stability. While the combination of `vector_size=1000 and min_count=10` achieves the highest overall performance (accuracy 86.88%), its improvement over `vector_size=500 and min_count=8` is modest (≈0.6%) but still nearly doubles the training time. To balance performance and computational efficiency, the configuration `vector_size=500, min_count=8` is adopted for subsequent classifier comparisons, whereas the best-performing configuration (`vector_size=1000, min_count=10`) is reserved for the final evaluation.

In [52]:
# Fifth attempt - Reassess performance stability
params_w2v = {
    'vector_size': [300, 500, 1000],
    'min_count': [8, 10],
    'sg': [1],
    'epochs': [10]
}

results_df = compare_w2v_params(params_w2v=params_w2v, classifier=LogisticRegression(max_iter=1000, random_state=42))

Training Word2Vec models...:   0%|                                            | 0/6 [00:00<?, ?it/s]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Training Word2Vec models...:  17%|██████                              | 1/6 [00:30<02:31, 30.33s/it]Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_flo

In [55]:
show_styled_df(results_df.sort_values(['accuracy'], ascending=False))

,vector_size,min_count,sg,epochs,embedding,classifier,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
5,1000,10,1,10,Word2Vec,LogisticRegression,86.88%,89.58%,88.74%,90.44%,82.30%,83.65%,81.00%,85.94%
4,1000,8,1,10,Word2Vec,LogisticRegression,86.68%,89.44%,88.46%,90.45%,81.98%,83.57%,80.45%,85.71%
2,500,8,1,10,Word2Vec,LogisticRegression,86.23%,89.08%,88.06%,90.14%,81.34%,83.00%,79.75%,85.21%
3,500,10,1,10,Word2Vec,LogisticRegression,86.14%,89.01%,88.02%,90.02%,81.24%,82.82%,79.71%,85.12%
1,300,10,1,10,Word2Vec,LogisticRegression,85.72%,88.73%,87.38%,90.12%,80.54%,82.74%,78.45%,84.63%
0,300,8,1,10,Word2Vec,LogisticRegression,85.51%,88.54%,87.32%,89.80%,80.29%,82.28%,78.41%,84.42%


### 2.2.6 Train–Test data partitioning
Since GridSearchCV internally performs cross-validation to tune hyperparameters, a two-way split of the dataset is sufficient. Therefore, the data is divided into `X_train` and `X_test` with a 70:30 ratio.

In [ ]:
train_df = pd.read_csv('../data/raw/ai_hum_text.csv')
X = train_df['Text']
y = train_df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

tokenized_train = [gensim_tokenize(s) for s in X_train]
tokenized_test  = [gensim_tokenize(s) for s in X_test]

In [7]:
y.value_counts()

Class
AI       62701
Human    37860
Name: count, dtype: int64

## 2.3 Classifier tuning

In [8]:
def build_w2v_features(w2v_params):
    """
    Train a Word2Vec model once and return standardized sentence vectors.

    Args:
        w2v_params:      dict of Word2Vec kwargs, e.g. {'vector_size':500,'min_count':8,'sg':1,'epochs':10}.

    Returns:
        Xtr_s: np.ndarray train features
        Xts_s: np.ndarray test features
    """
    fixed_w2v = Word2Vec(
        sentences=tokenized_train,
        seed=42,
        workers=1,  # for more reproducible training
        **w2v
    )
    
    Xtr = vectorize_texts(fixed_w2v, tokenized_train)
    Xts = vectorize_texts(fixed_w2v, tokenized_test)
    
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = scaler.fit_transform(Xtr)
    Xts_s = scaler.transform(Xts)
    return Xtr_s, Xts_s

In [12]:
w2v = {
    'vector_size': 500,
    'min_count': 8,
    'sg': 1,
    'epochs': 10
}

Xtr_s, Xts_s = build_w2v_features(w2v_params=w2v)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


### 2.3.1 LogisticRegression

Logistic Regression was one of the best-performing classifiers in Assignment 1.Therefore, it was included here to examine whether it can also achieve strong performance when combined with Word2Vec embeddings.

Several combinations of parameters were tested to understand how different solvers and regularization methods influence the model’s performance.

**1. `solver='lbfgs'`**

This solver only supports the L2 penalty, so tuning mainly focused on different values of `C` (regularization strength) and `class_weight` (None or balanced) to examine how stronger or weaker regularization, as well as class imbalance handling, affect performance.

**2. `solver='saga'`**

The saga solver supports both L1 and elasticnet penalties, allowing experiments that combine different `l1_ratio` values for elasticnet. Similar to the previous solver, `C` and `class_weight` were also included in the search space.

Since these combinations significantly increase the total number of configurations, `RandomizedSearchCV` with `n_iter=20` was used to limit the search while maintaining a fair exploration of the space.

Overall, the results show that the performance differences across these configurations are relatively small as both accuracy and macro-F1 vary by less than 1%. The main variation lies in how the models balance AI vs. Human prediction trade-offs. For instance, configurations with higher AI precision reduce the chance of misclassifying Human-written texts as AI-generated, which could be preferable in some use cases.

However, the runtime difference is substantial. Configurations using `solver='saga'` required 40–100× longer training time compared to `lbfgs`, despite achieving nearly identical performance.

Considering both effectiveness and computational efficiency, the configuration `{'solver': 'lbfgs', 'penalty': 'l2', 'C': 1, 'class_weight': None, 'max_iter': 2000}` is considered the most practical choice for subsequent experiments. While C values of 0.1 and 0.3 were not explicitly tested, the similar performance observed among C = 1, 3, and 10 suggests that the regularization strength has limited influence on the model performance. Hence, further exploration of C values was omitted for efficiency.

In [62]:
param_grid = [
    {
        'solver': ['lbfgs'],            
        'penalty': ['l2'],
        'C': [0.1, 0.3, 1, 3, 10],      
        'class_weight': [None, 'balanced'],
        'max_iter': [2000]
    },
    {
        'solver': ['saga'],
        'penalty': ['l1', 'elasticnet'],
        'l1_ratio': [0.1, 0.5, 0.9],    
        'C': [0.3, 1, 3],
        'class_weight': [None, 'balanced'],
        'max_iter': [5000]
    }
]

search = random_search_classifier(classifier=LogisticRegression(random_state=42), 
                                  clf_param_grid=param_grid,
                                  X_train=Xtr_s,
                                  y_train=y_train,
                                  n_iter=20)

In [81]:
log_search_results = pd.DataFrame(search.cv_results_)
show_search_results(log_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'solver': 'saga', 'penalty': 'elasticnet', 'max_iter': 5000, 'l1_ratio': 0.5, 'class_weight': None, 'C': 0.3}",717.994297,1.553211,86.58%,89.29%,88.84%,89.74%,82.02%,82.73%,81.33%,85.66%
1,"{'solver': 'saga', 'penalty': 'elasticnet', 'max_iter': 5000, 'l1_ratio': 0.9, 'class_weight': None, 'C': 0.3}",666.864186,1.285238,86.54%,89.26%,88.81%,89.73%,81.97%,82.69%,81.27%,85.62%
2,"{'solver': 'saga', 'penalty': 'l1', 'max_iter': 5000, 'l1_ratio': 0.5, 'class_weight': None, 'C': 0.3}",708.871077,1.572516,86.53%,89.25%,88.80%,89.71%,81.96%,82.67%,81.27%,85.61%
3,"{'solver': 'saga', 'penalty': 'l1', 'max_iter': 5000, 'l1_ratio': 0.9, 'class_weight': None, 'C': 1}",1257.718216,1.900604,86.53%,89.25%,88.82%,89.68%,81.97%,82.63%,81.31%,85.61%
4,"{'solver': 'saga', 'penalty': 'l1', 'max_iter': 5000, 'l1_ratio': 0.5, 'class_weight': None, 'C': 1}",1061.723909,1.714137,86.53%,89.25%,88.82%,89.68%,81.97%,82.63%,81.31%,85.61%
5,"{'solver': 'saga', 'penalty': 'elasticnet', 'max_iter': 5000, 'l1_ratio': 0.5, 'class_weight': None, 'C': 1}",1356.282426,1.972424,86.52%,89.24%,88.81%,89.68%,81.95%,82.63%,81.28%,85.59%
6,"{'solver': 'saga', 'penalty': 'elasticnet', 'max_iter': 5000, 'l1_ratio': 0.1, 'class_weight': None, 'C': 3}",1200.069988,1.681313,86.50%,89.22%,88.80%,89.66%,81.92%,82.59%,81.27%,85.57%
7,"{'solver': 'saga', 'penalty': 'elasticnet', 'max_iter': 5000, 'l1_ratio': 0.9, 'class_weight': None, 'C': 3}",1533.330106,2.591330,86.49%,89.22%,88.78%,89.66%,81.91%,82.59%,81.24%,85.56%
8,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
9,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 3}",25.676128,1.982835,86.46%,89.19%,88.75%,89.64%,81.86%,82.56%,81.18%,85.53%


In [83]:
# For combinations of solver='lbfgs' and class_weight=None, the difference of C doesn't matter much
cond1 = log_search_results['param_solver'] == 'lbfgs'
show_search_results(log_search_results[cond1])

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
1,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 3}",25.676128,1.982835,86.46%,89.19%,88.75%,89.64%,81.86%,82.56%,81.18%,85.53%
2,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 10}",24.998395,2.061522,86.45%,89.18%,88.75%,89.62%,81.86%,82.53%,81.19%,85.52%
3,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': 'balanced', 'C': 0.3}",21.565393,2.013584,85.99%,88.36%,91.70%,85.25%,82.42%,78.12%,87.22%,85.39%
4,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': 'balanced', 'C': 10}",17.105437,1.541940,85.99%,88.36%,91.68%,85.27%,82.42%,78.14%,87.19%,85.39%


In [133]:
best_results = pd.DataFrame()
best_log = log_search_results[log_search_results['params'] == {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}]
best_log.insert(0, 'classifier', 'LogisticRegression')
best_results = pd.concat([best_results, best_log])

### 2.3.2 RidgeClassifier
RidgeClassifier is a linear model that applies L2 regularization (ridge penalty) to prevent overfitting by shrinking model coefficients. As it provides stable performance when features are highly correlated or numerous, it was also considered a suitable candidate for evaluation in this experiment.

Different combinations of `class_weight` and `alpha` are tested on RidgeClassifier. The results are similar to those obtained from LogisticRegression. When `class_weight='balanced'`, the model shows higher AI precision but lower Human precision, as it tends to classify more texts as human-generated. Overall, using `class_weight=None` yields higher accuracy and macro F1, providing a more balanced performance across all metrics. Smaller alpha values lead to slightly better results, but the difference is negligible. Since the overall performance is slightly below that of LogisticRegression, further exploration was omitted.

In [88]:
param_grid = {
    'alpha': [0.1, 1.0, 10.0],
    'class_weight': [None, 'balanced']
}
ridge_search = random_search_classifier(classifier=RidgeClassifier(random_state=42),
                                        clf_param_grid=param_grid,
                                        X_train=Xtr_s,
                                        y_train=y_train,
                                        n_iter=6)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


In [99]:
ridge_search_results = pd.DataFrame(ridge_search.cv_results_)
show_search_results(ridge_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%
1,"{'class_weight': None, 'alpha': 1.0}",4.551754,1.448313,86.33%,89.11%,88.56%,89.66%,81.66%,82.51%,80.83%,85.38%
2,"{'class_weight': None, 'alpha': 10.0}",2.603614,1.415178,86.32%,89.10%,88.56%,89.64%,81.65%,82.50%,80.82%,85.38%
3,"{'class_weight': 'balanced', 'alpha': 10.0}",1.534151,1.155157,85.50%,87.83%,92.11%,83.93%,82.06%,76.80%,88.09%,84.95%
4,"{'class_weight': 'balanced', 'alpha': 1.0}",2.523035,1.454350,85.50%,87.83%,92.10%,83.94%,82.06%,76.81%,88.07%,84.94%
5,"{'class_weight': 'balanced', 'alpha': 0.1}",5.556357,1.466348,85.49%,87.83%,92.09%,83.94%,82.05%,76.81%,88.06%,84.94%


In [134]:
best_ridge = ridge_search_results[ridge_search_results['params'] == {'class_weight': None, 'alpha': 0.1}]
best_ridge.insert(0, 'classifier', 'RidgeClassifier')
best_results = pd.concat([best_results, best_ridge])
show_search_results(best_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LogisticRegression,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
1,RidgeClassifier,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%


### 2.3.3 LinearSVC
Linear SVC is a linear Support Vector Machine that finds the maximum-margin hyperplane between classes. It is robust to outliers and often achieves high accuracy on high-dimensional textual data.

In the first round of experiments, different combinations of `loss`, `C`, and `class_weight` were tested. The results show that using `loss='hinge'` with `class_weight=None` achieves higher accuracy and macro F1-score when `C=0.1`. Therefore, smaller values of `C` were further explored in the second attempt. However, the performance declines as C approaches 0, indicating that too strong regularization leads to underfitting. Overall, the configuration `{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}` provides the best balance between accuracy and stability.

In [96]:
# First attempt
param_grid = [
    {
        'loss': ['hinge'],
        'dual': [True],
        'C': [0.1, 1, 3, 10],
        'class_weight': [None, 'balanced'],
    },
    {
        'loss': ['squared_hinge'],
        'dual': [False],
        'C': [0.1, 1, 3, 10],
        'class_weight': [None, 'balanced'],
    },
]
svc_search = random_search_classifier(classifier=LinearSVC(random_state=42, max_iter=3000),
                                      clf_param_grid=param_grid,
                                      X_train=Xtr_s,
                                      y_train=y_train,
                                      n_iter=8)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/s

In [98]:
svc_search_results = pd.DataFrame(svc_search.cv_results_)
show_search_results(svc_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
1,"{'loss': 'squared_hinge', 'dual': False, 'class_weight': None, 'C': 10}",131.719089,2.624357,86.50%,89.21%,88.90%,89.52%,81.96%,82.45%,81.48%,85.59%
2,"{'loss': 'squared_hinge', 'dual': False, 'class_weight': None, 'C': 0.1}",91.642456,2.268751,86.49%,89.20%,88.89%,89.52%,81.95%,82.44%,81.47%,85.58%
3,"{'loss': 'hinge', 'dual': True, 'class_weight': 'balanced', 'C': 0.1}",91.745410,2.125254,85.93%,88.29%,91.80%,85.03%,82.39%,77.91%,87.42%,85.34%
4,"{'loss': 'squared_hinge', 'dual': False, 'class_weight': 'balanced', 'C': 1}",113.136028,2.238218,85.89%,88.25%,91.75%,85.01%,82.33%,77.87%,87.33%,85.29%
5,"{'loss': 'squared_hinge', 'dual': False, 'class_weight': 'balanced', 'C': 3}",130.961695,2.595333,85.89%,88.25%,91.74%,85.01%,82.33%,77.87%,87.33%,85.29%
6,"{'loss': 'squared_hinge', 'dual': False, 'class_weight': 'balanced', 'C': 0.1}",98.048838,1.587652,85.88%,88.24%,91.75%,84.99%,82.32%,77.84%,87.35%,85.28%
7,"{'loss': 'hinge', 'dual': True, 'class_weight': 'balanced', 'C': 3}",521.459385,0.991086,85.71%,88.13%,91.40%,85.10%,82.05%,77.85%,86.73%,85.09%


In [136]:
# Second attempt
param_grid = {
        'loss': ['hinge'],
        'dual': [True],
        'C': [0.001, 0.01, 0.1],
        'class_weight': [None],
}
svc_search = random_search_classifier(classifier=LinearSVC(random_state=42, max_iter=3000),
                                      clf_param_grid=param_grid,
                                      X_train=Xtr_s,
                                      y_train=y_train,
                                      n_iter=8)
svc_search_results = pd.DataFrame(svc_search.cv_results_)
show_search_results(svc_search_results)

/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=8. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 3 candidates, totalling 9 fits


/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/COMP47750/lib/python3.13/site-packages/s

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",53.297994,1.091918,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
1,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.01}",19.732592,1.754698,86.47%,89.17%,89.03%,89.31%,81.99%,82.20%,81.77%,85.58%
2,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.001}",11.166161,1.847071,86.21%,88.90%,89.26%,88.54%,81.81%,81.27%,82.35%,85.35%


In [135]:
best_svc = svc_search_results[svc_search_results['params'] == {'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}]
best_svc.insert(0, 'classifier', 'LinearSVC')
best_results = pd.concat([best_results, best_svc])
show_search_results(best_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
1,LogisticRegression,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
2,RidgeClassifier,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%


### 2.3.4 SGDClassifier
SGD Classifier is a linear model trained via Stochastic Gradient Descent, supporting various loss functions (e.g., hinge, log). It offers computational efficiency and scalability for large-scale datasets.

In the first round of experiments, various combinations of `loss`, `alpha`, `penalty`, and `class_weight` were tested. The results indicate that using `penalty='l2'` with `class_weight=None` achieves higher accuracy and macro F1-score when `alpha=0.01`. Consequently, larger values of alpha were explored in the second attempt, but the performance declined as alpha increased. The loss function contributed little difference to the results. Overall, the configuration `{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}` offers the best trade-off between accuracy and stability.

While SGDClassifier shows competitive results, LinearSVC remains the overall best-performing model in this comparison.

In [141]:
param_grid = {
    'loss': ['log_loss', 'hinge'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'penalty': ['l2', 'elasticnet'],
    'l1_ratio': [0.15, 0.3, 0.5, 0.7],
    'class_weight': [None, 'balanced']
}

sgd_search = random_search_classifier(classifier=SGDClassifier(random_state=42, max_iter=1000),
                                      clf_param_grid=param_grid,
                                      X_train=Xtr_s,
                                      y_train=y_train,
                                      n_iter=20)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [142]:
sgd_search_results = pd.DataFrame(sgd_search.cv_results_)
show_search_results(sgd_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'penalty': 'l2', 'loss': 'log_loss', 'l1_ratio': 0.15, 'class_weight': None, 'alpha': 0.01}",3.163791,2.419434,86.16%,88.97%,88.42%,89.54%,81.43%,82.30%,80.57%,85.20%
1,"{'penalty': 'elasticnet', 'loss': 'log_loss', 'l1_ratio': 0.15, 'class_weight': None, 'alpha': 0.001}",14.323874,1.622454,85.96%,88.82%,88.18%,89.47%,81.12%,82.14%,80.13%,84.97%
2,"{'penalty': 'l2', 'loss': 'hinge', 'l1_ratio': 0.5, 'class_weight': None, 'alpha': 0.001}",8.097668,1.757248,85.84%,88.68%,88.42%,88.96%,81.10%,81.53%,80.69%,84.89%
3,"{'penalty': 'l2', 'loss': 'hinge', 'l1_ratio': 0.7, 'class_weight': 'balanced', 'alpha': 0.01}",3.074906,1.708758,85.52%,87.91%,91.64%,84.48%,81.94%,77.24%,87.24%,84.93%
4,"{'penalty': 'l2', 'loss': 'log_loss', 'l1_ratio': 0.15, 'class_weight': 'balanced', 'alpha': 0.01}",7.433481,2.475238,85.49%,87.91%,91.44%,84.65%,81.84%,77.37%,86.87%,84.88%
5,"{'penalty': 'elasticnet', 'loss': 'log_loss', 'l1_ratio': 0.3, 'class_weight': None, 'alpha': 0.01}",6.681461,1.780451,85.49%,88.51%,87.42%,89.62%,80.31%,82.06%,78.64%,84.41%
6,"{'penalty': 'l2', 'loss': 'log_loss', 'l1_ratio': 0.15, 'class_weight': None, 'alpha': 0.0001}",24.539604,2.953052,85.46%,88.35%,88.32%,88.38%,80.68%,80.73%,80.64%,84.52%
7,"{'penalty': 'l2', 'loss': 'log_loss', 'l1_ratio': 0.3, 'class_weight': None, 'alpha': 0.0001}",22.217954,2.138015,85.46%,88.35%,88.32%,88.38%,80.68%,80.73%,80.64%,84.52%
8,"{'penalty': 'l2', 'loss': 'log_loss', 'l1_ratio': 0.7, 'class_weight': None, 'alpha': 0.0001}",23.724829,1.921482,85.46%,88.35%,88.32%,88.38%,80.68%,80.73%,80.64%,84.52%
9,"{'penalty': 'elasticnet', 'loss': 'log_loss', 'l1_ratio': 0.3, 'class_weight': 'balanced', 'alpha': 0.001}",16.698613,1.056564,85.41%,87.88%,91.17%,84.81%,81.68%,77.45%,86.39%,84.78%


In [145]:
param_grid = {
    'loss': ['hinge', 'log_loss'],
    'alpha': [0.01, 0.1, 1],
    'penalty': ['l2'],
    'class_weight': [None]
}

sgd_search = random_search_classifier(classifier=SGDClassifier(random_state=42, max_iter=1000),
                                      clf_param_grid=param_grid,
                                      X_train=Xtr_s,
                                      y_train=y_train,
                                      n_iter=18)
sgd_search_results = pd.DataFrame(sgd_search.cv_results_)
show_search_results(sgd_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}",3.452634,2.514182,86.21%,88.96%,88.77%,89.16%,81.61%,81.91%,81.32%,85.29%
1,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 0.01}",3.370577,2.513036,86.16%,88.97%,88.42%,89.54%,81.43%,82.30%,80.57%,85.20%
2,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.1}",1.933152,1.824942,85.87%,88.72%,88.37%,89.07%,81.12%,81.66%,80.58%,84.92%
3,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 0.1}",2.268649,1.604555,85.54%,88.57%,87.30%,89.88%,80.32%,82.38%,78.35%,84.45%
4,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 1}",2.107868,1.594338,84.81%,88.04%,86.48%,89.68%,79.18%,81.84%,76.75%,83.61%
5,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 1}",1.996269,0.987165,83.63%,87.12%,85.65%,88.80%,77.46%,80.56%,75.08%,82.29%


In [146]:
best_sgd = sgd_search_results[sgd_search_results['params'] == {'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}]
best_sgd.insert(0, 'classifier', 'SGDClassifier')
best_results = pd.concat([best_results, best_sgd])
show_search_results(best_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
1,LogisticRegression,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
2,RidgeClassifier,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%
3,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}",3.452634,2.514182,86.21%,88.96%,88.77%,89.16%,81.61%,81.91%,81.32%,85.29%


### 2.3.5 HistGradientBoostingClassifier
HistGradientBoostingClassifier is an ensemble method that builds decision-tree models sequentially using gradient boosting with histogram-based optimization.

In the first round of experiments, various combinations of `learning_rate`, `max_iter`, `max_depth`, `l2_regularization`, `max_leaf_nodes` were tested. These hypermeters control the model's learning behavior at different levels:
- `learning_rate` adjusts how aggressively the model updates each tree’s contribution;
- `max_iter` determines the number of boosting stages;
- `max_depth` and `max_leaf_nodes` control the tree complexity;
- `l2_regularization` helps prevent overfitting by penalising large leaf weights.
    
The fitting time for each configuration was approximately 2–5 minutes, and the performance was generally comparable to that of the other classifiers. However, during the first round of experiments, the best-performing parameter combinations were found near the boundary values. Therefore, a second round of tuning was conducted to explore whether extending these boundaries could further improve the results. Although the performance did improve slightly, the difference compared to previous classifiers was minimal, while the training time increased significantly. This suggests that despite its potential for optimization, the model may not be an ideal candidate due to its relatively high computational cost.

In [154]:
from sklearn.ensemble import HistGradientBoostingClassifier

param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_iter': [200, 300, 500],
    'max_depth': [6, 8, 10],
    'l2_regularization': [0.0, 0.5, 1.0],
    'max_leaf_nodes': [31, 63]
}

hgb_search = random_search_classifier(
    classifier=HistGradientBoostingClassifier(random_state=42, early_stopping=True),
    clf_param_grid=param_grid,
    X_train=Xtr_s,
    y_train=y_train,
    n_iter=18
)

hgb_search_results = pd.DataFrame(hgb_search.cv_results_)
show_search_results(hgb_search_results)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=6, max_iter=200, max_leaf_nodes=63; total time= 2.6min
[CV] END l2_regularization=0.5, learning_rate=0.05, max_depth=6, max_iter=200, max_leaf_nodes=63; total time= 2.3min
[CV] END l2_regularization=0.0, learning_rate=0.2, max_depth=10, max_iter=300, max_leaf_nodes=63; total time= 2.7min
[CV] END l2_regularization=1.0, learning_rate=0.2, max_depth=6, max_iter=200, max_leaf_nodes=31; total time= 2.0min
[CV] END l2_regularization=0.0, learning_rate=0.05, max_depth=10, max_iter=300, max_leaf_nodes=63; total time= 5.4min
[CV] END l2_regularization=1.0, learning_rate=0.1, max_depth=10, max_iter=500, max_leaf_nodes=63; total time= 3.3min


,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 1.0}",192.954003,3.741642,85.99%,88.84%,88.31%,89.36%,81.21%,82.03%,80.41%,85.02%
1,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 6, 'learning_rate': 0.1, 'l2_regularization': 1.0}",175.909870,10.660520,85.98%,88.85%,88.14%,89.56%,81.13%,82.24%,80.05%,84.99%
2,"{'max_leaf_nodes': 63, 'max_iter': 300, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 0.0}",301.500415,11.283782,85.98%,88.83%,88.24%,89.43%,81.17%,82.10%,80.27%,85.00%
3,"{'max_leaf_nodes': 31, 'max_iter': 200, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 0.0}",116.752443,6.084305,85.80%,88.69%,88.12%,89.27%,80.94%,81.83%,80.06%,84.81%
4,"{'max_leaf_nodes': 31, 'max_iter': 200, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 0.5}",155.275058,6.845906,85.80%,88.69%,88.10%,89.28%,80.93%,81.84%,80.04%,84.81%
5,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 8, 'learning_rate': 0.1, 'l2_regularization': 0.0}",227.357922,14.009851,85.71%,88.61%,88.07%,89.15%,80.83%,81.66%,80.01%,84.72%
6,"{'max_leaf_nodes': 31, 'max_iter': 200, 'max_depth': 8, 'learning_rate': 0.1, 'l2_regularization': 0.0}",125.959147,7.923663,85.70%,88.60%,88.07%,89.15%,80.82%,81.66%,80.00%,84.71%
7,"{'max_leaf_nodes': 63, 'max_iter': 200, 'max_depth': 6, 'learning_rate': 0.1, 'l2_regularization': 0.0}",156.815088,8.509222,85.67%,88.61%,87.88%,89.35%,80.71%,81.86%,79.59%,84.66%
8,"{'max_leaf_nodes': 31, 'max_iter': 200, 'max_depth': 6, 'learning_rate': 0.2, 'l2_regularization': 1.0}",105.036443,7.348912,85.57%,88.49%,88.02%,88.97%,80.66%,81.40%,79.94%,84.58%
9,"{'max_leaf_nodes': 31, 'max_iter': 300, 'max_depth': 10, 'learning_rate': 0.2, 'l2_regularization': 1.0}",112.550793,6.446724,85.55%,88.47%,88.03%,88.91%,80.64%,81.33%,79.97%,84.56%


In [158]:
# Higher max_depth may worth a try
cond1 = hgb_search_results['param_max_leaf_nodes'] == 63
cond2 = hgb_search_results['param_max_iter'] == 500
show_search_results(hgb_search_results[cond1 & cond2])

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 1.0}",192.954003,3.741642,85.99%,88.84%,88.31%,89.36%,81.21%,82.03%,80.41%,85.02%
1,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 6, 'learning_rate': 0.1, 'l2_regularization': 1.0}",175.909870,10.660520,85.98%,88.85%,88.14%,89.56%,81.13%,82.24%,80.05%,84.99%
2,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 8, 'learning_rate': 0.1, 'l2_regularization': 0.0}",227.357922,14.009851,85.71%,88.61%,88.07%,89.15%,80.83%,81.66%,80.01%,84.72%
3,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 8, 'learning_rate': 0.2, 'l2_regularization': 0.5}",134.150349,6.181916,85.16%,88.19%,87.54%,88.86%,80.05%,81.07%,79.05%,84.12%


In [160]:
# Second attempt
param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_iter': [500, 800],
    'max_depth': [10, None],
    'l2_regularization': [1.0, 3.0, 5.0],
    'max_leaf_nodes': [63, 127, 255]
}

hgb_search = random_search_classifier(
    classifier=HistGradientBoostingClassifier(random_state=42, early_stopping=True),
    clf_param_grid=param_grid,
    X_train=Xtr_s,
    y_train=y_train,
    n_iter=18
)

hgb_search_results = pd.DataFrame(hgb_search.cv_results_)
show_search_results(hgb_search_results)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=10, max_iter=800, max_leaf_nodes=127; total time=11.9min
[CV] END l2_regularization=5.0, learning_rate=0.05, max_depth=10, max_iter=500, max_leaf_nodes=255; total time=19.3min
[CV] END l2_regularization=1.0, learning_rate=0.1, max_depth=10, max_iter=500, max_leaf_nodes=63; total time= 4.6min
[CV] END l2_regularization=3.0, learning_rate=0.1, max_depth=None, max_iter=800, max_leaf_nodes=255; total time=10.4min
[CV] END l2_regularization=3.0, learning_rate=0.1, max_depth=None, max_iter=800, max_leaf_nodes=63; total time= 5.1min
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=10, max_iter=800, max_leaf_nodes=255; total time=15.3min
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=10, max_iter=800, max_leaf_nodes=127; total time=11.8min
[CV] END l2_regularization=5.0, learning_rate=0.05, max_depth=10, max_iter=500, max_leaf_nodes=255; total t

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}",593.634063,27.035900,86.35%,89.11%,88.64%,89.58%,81.71%,82.44%,80.99%,85.41%
1,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 1.0}",578.133706,27.891556,86.28%,89.06%,88.52%,89.61%,81.59%,82.44%,80.76%,85.33%
2,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 5.0}",596.647564,24.404299,86.26%,89.04%,88.57%,89.52%,81.59%,82.33%,80.87%,85.32%
3,"{'max_leaf_nodes': 127, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 5.0}",522.049283,9.005515,86.12%,88.95%,88.33%,89.58%,81.35%,82.33%,80.39%,85.15%
4,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.1, 'l2_regularization': 3.0}",299.808368,12.763122,86.10%,88.92%,88.37%,89.47%,81.34%,82.20%,80.50%,85.13%
5,"{'max_leaf_nodes': 127, 'max_iter': 800, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 3.0}",777.000415,19.669948,86.08%,88.91%,88.30%,89.53%,81.30%,82.25%,80.36%,85.10%
6,"{'max_leaf_nodes': 127, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 3.0}",769.427408,25.076266,86.07%,88.90%,88.33%,89.48%,81.30%,82.19%,80.42%,85.10%
7,"{'max_leaf_nodes': 127, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 3.0}",758.940844,23.609397,86.07%,88.90%,88.33%,89.48%,81.30%,82.19%,80.42%,85.10%
8,"{'max_leaf_nodes': 127, 'max_iter': 800, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 1.0}",684.863604,20.550803,86.00%,88.87%,88.16%,89.58%,81.16%,82.28%,80.08%,85.01%
9,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 1.0}",249.664063,10.586325,85.99%,88.84%,88.31%,89.36%,81.21%,82.03%,80.41%,85.02%


[CV] END l2_regularization=1.0, learning_rate=0.1, max_depth=None, max_iter=500, max_leaf_nodes=63; total time= 4.6min
[CV] END l2_regularization=1.0, learning_rate=0.1, max_depth=None, max_iter=500, max_leaf_nodes=63; total time= 5.2min
[CV] END l2_regularization=3.0, learning_rate=0.05, max_depth=10, max_iter=800, max_leaf_nodes=127; total time=13.7min
[CV] END l2_regularization=3.0, learning_rate=0.05, max_depth=None, max_iter=800, max_leaf_nodes=127; total time=12.6min
[CV] END l2_regularization=3.0, learning_rate=0.1, max_depth=None, max_iter=800, max_leaf_nodes=255; total time= 9.0min
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=None, max_iter=800, max_leaf_nodes=63; total time= 9.8min
[CV] END l2_regularization=1.0, learning_rate=0.1, max_depth=None, max_iter=800, max_leaf_nodes=127; total time= 7.2min
[CV] END l2_regularization=5.0, learning_rate=0.05, max_depth=10, max_iter=500, max_leaf_nodes=127; total time= 9.0min
[CV] END l2_regularization=5.0, learning_ra

In [161]:
best_hgb = hgb_search_results[hgb_search_results['params'] == {'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}]
best_hgb.insert(0, 'classifier', 'HistGradientBoostingClassifier')
best_results = pd.concat([best_results, best_hgb])
show_search_results(best_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
1,LogisticRegression,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
2,HistGradientBoostingClassifier,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}",593.634063,27.035900,86.35%,89.11%,88.64%,89.58%,81.71%,82.44%,80.99%,85.41%
3,RidgeClassifier,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%
4,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}",3.452634,2.514182,86.21%,88.96%,88.77%,89.16%,81.61%,81.91%,81.32%,85.29%


### 2.3.6 XGBClassifier

XGBClassifier is an optimized implementation of Gradient Boosting that incorporates both regularization and parallelization. It offers strong generalization capability and is considered a highly effective modern boosting widely applied to both text and tabular classification tasks. Since it only accepts binary numerical targets (0 and 1), the label values were converted accordingly before training.

Similar to HistGradientBoostingClassifier, XGBClassifier required substantially longer training time. The final optimized configuration `{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}` was found in the second round of hyperparameter tuning. Although increasing `n_estimators` from 800 to 1000 did lead to a small performance improvement, the gain was marginal. Therefore, further parameter tuning was not pursued. While the final configuration outperformed LinearSVC, the improvement was subtle and required significantly more computation, which is an important consideration when selecting the final model.

In [166]:
y_train_num = (y_train == 'AI').astype(int)

In [167]:
param_grid = {
    'learning_rate': [0.05, 0.1],          
    'n_estimators': [300, 500],            
    'max_depth': [4, 6],                   
    'subsample': [0.8],                    
    'colsample_bytree': [0.8],             
    'reg_lambda': [0.5, 1.0, 2.0],
    'reg_alpha': [0.0, 0.1]
}

xgb_search = random_search_classifier(
    classifier=XGBClassifier(
        random_state=42,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1,                         
        verbosity=0
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_s,
    y_train=y_train_num,                   
    n_iter=12,                             
    ai_label=1, 
    human_label=0
)

ERROR! Session/line number was not unique in database. History logging moved to new session 133
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=4, n_estimators=300, reg_alpha=0.1, reg_lambda=0.5, subsample=0.8; total time= 2.6min
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=4, n_estimators=300, reg_alpha=0.0, reg_lambda=0.5, subsample=0.8; total time= 2.5min
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=300, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time= 4.0min
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=500, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time= 6.0min
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=300, reg_alpha=0.1, reg_lambda=0.5, subsample=0.8; total time=  53.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=4, n_estimators=300, reg_alpha=0.1, reg_lambda=0.5,

In [168]:
xgb_search_results = pd.DataFrame(xgb_search.cv_results_)
show_search_results(xgb_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",385.535457,1.867720,86.29%,89.07%,88.54%,89.61%,81.61%,82.44%,80.79%,85.34%
1,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.8}",359.713789,1.262463,86.27%,89.06%,88.45%,89.69%,81.55%,82.52%,80.60%,85.31%
2,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",238.581240,1.039986,86.06%,88.91%,88.24%,89.59%,81.25%,82.31%,80.22%,85.08%
3,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",275.355045,2.113121,86.03%,88.87%,88.27%,89.48%,81.23%,82.17%,80.31%,85.05%
4,"{'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.0, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",138.096282,0.565063,85.81%,88.73%,87.90%,89.57%,80.85%,82.16%,79.59%,84.79%
5,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",123.364832,0.567898,85.80%,88.70%,87.97%,89.45%,80.87%,82.03%,79.74%,84.79%
6,"{'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.1, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",151.782455,1.571570,85.78%,88.69%,87.95%,89.43%,80.84%,82.00%,79.71%,84.76%
7,"{'subsample': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.8}",160.817192,0.412885,85.75%,88.67%,87.90%,89.45%,80.79%,82.00%,79.61%,84.73%
8,"{'subsample': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",151.387129,1.767028,85.70%,88.63%,87.89%,89.38%,80.74%,81.91%,79.61%,84.69%
9,"{'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.0, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.8}",246.227805,1.069449,85.64%,88.60%,87.71%,89.51%,80.60%,82.02%,79.22%,84.60%


In [170]:
param_grid = {
    'learning_rate': [0.1, 0.2],          
    'n_estimators': [500, 800, 1000],            
    'max_depth': [6, 8, 10],                   
    'subsample': [0.8],                    
    'colsample_bytree': [0.8],             
    'reg_lambda': [1.0],
    'reg_alpha': [0.0]
}

xgb_search = random_search_classifier(
    classifier=XGBClassifier(
        random_state=42,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1,                         
        verbosity=0
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_s,
    y_train=y_train_num,                   
    n_iter=12,                             
    ai_label=1, 
    human_label=0
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=800, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time=13.2min
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=8, n_estimators=1000, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time=21.7min
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=10, n_estimators=800, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time=24.5min
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=1000, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time= 8.8min
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=800, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time=13.3min
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=8, n_estimators=500, reg_alpha=0.0, reg_lambda=1.0, subsample=0.8; total time=16.9min
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=

In [171]:
xgb_search_results = pd.DataFrame(xgb_search.cv_results_)
show_search_results(xgb_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",496.809922,2.458532,86.72%,89.41%,88.93%,89.90%,82.21%,82.96%,81.47%,85.81%
1,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 800, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",792.606497,3.220244,86.63%,89.34%,88.84%,89.84%,82.08%,82.86%,81.32%,85.71%
2,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",1337.047255,3.241504,86.62%,89.35%,88.68%,90.03%,82.00%,83.06%,80.97%,85.68%
3,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 0.8}",510.116102,2.532670,86.58%,89.30%,88.82%,89.78%,82.02%,82.77%,81.29%,85.66%
4,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 10, 'learning_rate': 0.2, 'colsample_bytree': 0.8}",498.263902,1.148088,86.46%,89.23%,88.53%,89.94%,81.78%,82.89%,80.71%,85.51%
5,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",2451.625719,4.753259,86.43%,89.21%,88.49%,89.93%,81.74%,82.87%,80.64%,85.47%
6,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.2, 'colsample_bytree': 0.8}",832.569017,14.955328,86.41%,89.17%,88.61%,89.74%,81.76%,82.65%,80.89%,85.47%
7,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.2, 'colsample_bytree': 0.8}",1502.678001,3.838414,86.34%,89.14%,88.38%,89.91%,81.59%,82.80%,80.43%,85.37%
8,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",947.443473,5.548292,86.31%,89.11%,88.36%,89.88%,81.55%,82.75%,80.39%,85.33%
9,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",562.947264,3.560866,86.29%,89.07%,88.54%,89.61%,81.61%,82.44%,80.79%,85.34%


In [184]:
best_xgb = xgb_search_results[xgb_search_results['params'] == {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}]
best_xgb.insert(0, 'classifier', 'XGBClassifier')
best_results = pd.concat([best_results, best_xgb])
show_search_results(best_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,XGBClassifier,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",496.809922,2.458532,86.72%,89.41%,88.93%,89.90%,82.21%,82.96%,81.47%,85.81%
1,LinearSVC,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
2,LogisticRegression,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
3,HistGradientBoostingClassifier,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}",593.634063,27.035900,86.35%,89.11%,88.64%,89.58%,81.71%,82.44%,80.99%,85.41%
4,RidgeClassifier,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%
5,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}",3.452634,2.514182,86.21%,88.96%,88.77%,89.16%,81.61%,81.91%,81.32%,85.29%


### 2.3.7 LightGBM

LightGBM is a gradient boosting ensemble model designed for high efficiency and scalability. It uses a leaf-wise tree growth strategy and histogram-based optimization, which allow it to train significantly faster than traditional Gradient Boosting or XGBoost while achieving comparable accuracy. Due to its speed, strong generalization ability, and suitability for dense feature representations, LightGBM was included as a final ensemble-style model to evaluate whether a faster boosting approach could outperform or complement the previously tested models.

Since model fitting is computationally expensive, a stratified sample of 8,000 training instances was used for hyperparameter tuning and configuration comparison. The best-performing configuration was then retrained on the full dataset to allow a fair comparison with the other classifiers.

After two rounds of parameter tuning, the configuration `{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}` achieved the best performance and was subsequently retrained on the full training dataset for comparison with the previous classifiers.

In [18]:
# Generate a stratified sample of 8,000 training instances
X_small, _, y_small, _ = train_test_split(
    Xtr_s,
    y_train,
    train_size=8000,
    stratify=y_train,
    random_state=42
)

In [32]:
# First attempt
param_grid = {
    'learning_rate': [0.05, 0.1],
    'n_estimators': [300, 500, 800],
    'num_leaves': [31, 63, 127],
    'max_depth': [-1, 8, 10],
    'subsample': [0.7, 0.9],
    'feature_fraction': [0.7, 0.9],
    'reg_lambda': [0.5, 1.0, 2.0],
    'reg_alpha': [0.0, 0.1, 0.5]
}


gbm_search = random_search_classifier(
    classifier=LGBMClassifier(
        random_state=42,
        objective='binary',
        n_jobs=-1
    ),
    clf_param_grid=param_grid,
    X_train=X_small,
    y_train=y_small,       
    n_iter=12
)

In [21]:
gbm_results = pd.DataFrame(gbm_search.cv_results_)
show_search_results(gbm_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.9}",196.662544,1.604636,83.83%,87.21%,86.03%,88.41%,78.02%,79.90%,76.23%,82.61%
1,"{'subsample': 0.9, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.7}",168.024438,2.572523,83.64%,87.06%,85.87%,88.29%,77.75%,79.68%,75.93%,82.40%
2,"{'subsample': 0.9, 'reg_lambda': 0.5, 'reg_alpha': 0.0, 'num_leaves': 31, 'n_estimators': 300, 'max_depth': -1, 'learning_rate': 0.1, 'feature_fraction': 0.9}",59.596173,1.018608,83.64%,87.08%,85.77%,88.43%,77.69%,79.81%,75.70%,82.39%
3,"{'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.1, 'feature_fraction': 0.9}",119.070080,1.428217,83.56%,87.09%,85.33%,88.93%,77.37%,80.30%,74.67%,82.23%
4,"{'subsample': 0.7, 'reg_lambda': 0.5, 'reg_alpha': 0.0, 'num_leaves': 63, 'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.1, 'feature_fraction': 0.9}",113.281650,1.821661,83.44%,86.97%,85.36%,88.65%,77.27%,79.93%,74.80%,82.12%
5,"{'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 0.5, 'num_leaves': 63, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.05, 'feature_fraction': 0.7}",183.005579,0.543530,83.40%,86.90%,85.54%,88.31%,77.34%,79.56%,75.27%,82.12%
6,"{'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 0.1, 'num_leaves': 127, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.9}",340.862729,3.830748,83.38%,86.88%,85.51%,88.31%,77.30%,79.54%,75.20%,82.09%
7,"{'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}",168.358285,1.401392,83.20%,86.80%,85.11%,88.55%,76.91%,79.69%,74.34%,81.85%
8,"{'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.9}",233.756988,3.163706,83.09%,86.67%,85.22%,88.17%,76.87%,79.22%,74.67%,81.77%
9,"{'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.1, 'feature_fraction': 0.7}",63.859412,1.070898,83.04%,86.68%,84.90%,88.55%,76.64%,79.60%,73.90%,81.66%


In [31]:
# Second attempt
param_grid = {
    'learning_rate': [0.05],
    'n_estimators': [800],
    'num_leaves': [31, 63, 127],
    'max_depth': [-1, 8, 10],
    'subsample': [0.9],
    'feature_fraction': [0.9],
    'reg_lambda': [1.0],
    'reg_alpha': [0.5]
}

gbm_search = random_search_classifier(
    classifier=LGBMClassifier(
        random_state=42,
        objective='binary',
        n_jobs=-1
    ),
    clf_param_grid=param_grid,
    X_train=X_small,
    y_train=y_small,       
    n_iter=12
)

In [25]:
gbm_results = pd.DataFrame(gbm_search.cv_results_)
show_search_results(gbm_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}",146.557509,1.514085,84.00%,87.35%,86.13%,88.61%,78.23%,80.21%,76.36%,82.79%
1,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.05, 'feature_fraction': 0.9}",160.780010,1.518803,83.84%,87.21%,86.05%,88.41%,78.03%,79.90%,76.26%,82.62%
2,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.9}",198.829492,1.505115,83.83%,87.21%,86.03%,88.41%,78.02%,79.90%,76.23%,82.61%
3,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 63, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}",175.055084,1.805538,83.46%,86.97%,85.48%,88.51%,77.37%,79.79%,75.10%,82.17%
4,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 63, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.9}",234.914449,1.838862,83.44%,86.90%,85.72%,88.13%,77.47%,79.38%,75.66%,82.19%
5,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 63, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.05, 'feature_fraction': 0.9}",177.437862,0.712425,83.30%,86.82%,85.47%,88.21%,77.22%,79.39%,75.17%,82.02%
6,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.05, 'feature_fraction': 0.9}",100.698504,0.670816,83.26%,86.83%,85.25%,88.47%,77.05%,79.64%,74.63%,81.94%
7,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}",182.826213,1.854444,83.24%,86.80%,85.24%,88.43%,77.03%,79.59%,74.63%,81.92%
8,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 127, 'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.05, 'feature_fraction': 0.9}",231.175106,2.141286,83.09%,86.67%,85.22%,88.17%,76.87%,79.22%,74.67%,81.77%


In [29]:
# Apply the best configuration on the full training data
param_grid = {
    'learning_rate': [0.05],
    'n_estimators': [800],
    'num_leaves': [31],
    'max_depth': [8],
    'subsample': [0.9],
    'feature_fraction': [0.9],
    'reg_lambda': [1.0],
    'reg_alpha': [0.5]
}

gbm_search = random_search_classifier(
    classifier=LGBMClassifier(
        random_state=42,
        objective='binary',
        n_jobs=-1
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_s,
    y_train=y_train,       
    n_iter=12
)

In [30]:
gbm_results = pd.DataFrame(gbm_search.cv_results_)
show_search_results(gbm_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}",192.490183,7.400811,86.56%,89.26%,88.91%,89.62%,82.03%,82.58%,81.49%,85.65%


In [59]:
gbm_results.insert(0, 'classifier', 'LGBMClassifier')
best_results = pd.concat([best_results, gbm_results])

In [ ]:
best_results.to_csv('../data/interim/word2vec_500_best_results.csv', index=False)

### 2.3.8 Best-performing classifiers using Word2Vec embeddings
After evaluating eight classifiers across multiple hyperparameter configurations, XGBClassifier achieved the highest performance in the 3-fold cross-validation using Word2Vec embeddings. However, LinearSVC produced nearly identical accuracy and macro-F1 scores (within 0.2% of XGBClassifier) while requiring only about 1/6 of the training time. Given its substantially higher computational efficiency and only marginally lower performance, LinearSVC was selected as the final Word2Vec-based classifier and applied to the best-performing Word2Vec configuration identified in the previous section.

The best-performing Word2Vec configuration that offers both strong performance and good computational efficiency is `{'vector_size': 1000, 'min_count': 10, 'sg': 1, 'epochs': 10}`. Using this embedding, the LinearSVC classifier with the configuration `{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}` achieves the following results under 3-fold cross-validation on the training data:
- Accuracy: 86.90%
- AI F1: 89.51%
- AI Precision: 89.32%
- AI Recall: 89.71%
- Human F1: 82.53%
- Human Precision: 82.84%
- Human Recall: 82.24%
- Macro F1: 86.02%

In [36]:
show_search_results(best_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,XGBClassifier,"{'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",496.809922,2.458532,86.72%,89.41%,88.93%,89.90%,82.21%,82.96%,81.47%,85.81%
1,LinearSVC,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%
2,LGBMClassifier,"{'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.9}",192.490183,7.400811,86.56%,89.26%,88.91%,89.62%,82.03%,82.58%,81.49%,85.65%
3,LogisticRegression,"{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'class_weight': None, 'C': 1}",15.442982,1.515607,86.46%,89.19%,88.76%,89.64%,81.87%,82.55%,81.19%,85.53%
4,HistGradientBoostingClassifier,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}",593.634063,27.035900,86.35%,89.11%,88.64%,89.58%,81.71%,82.44%,80.99%,85.41%
5,RidgeClassifier,"{'class_weight': None, 'alpha': 0.1}",5.620996,1.447425,86.34%,89.11%,88.57%,89.66%,81.67%,82.51%,80.84%,85.39%
6,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}",3.452634,2.514182,86.21%,88.96%,88.77%,89.16%,81.61%,81.91%,81.32%,85.29%


In [86]:
w2v = {
    'vector_size': 1000,
    'min_count': 10,
    'sg': 1,
    'epochs': 10
}

Xtr_s, Xts_s = build_w2v_features(w2v_params=w2v)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [91]:
# The best configuration of LinearSVC
param_grid = {
    'loss': ['hinge'],
    'dual': [True],
    'class_weight': [None], 
    'C': [0.1]}
svc_search = random_search_classifier(classifier=LinearSVC(random_state=42, max_iter=3000),
                                      clf_param_grid=param_grid,
                                      X_train=Xtr_s,
                                      y_train=y_train,
                                      n_iter=8)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [94]:
# Previously used Word2Vec configuration
show_search_results(best_results[best_results['classifier']=='LinearSVC'])

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",89.259389,2.811975,86.58%,89.26%,89.10%,89.42%,82.12%,82.38%,81.88%,85.69%


In [95]:
# Best-performing Word2Vec configuration
w2v_svc_search_results = pd.DataFrame(svc_search.cv_results_)
show_search_results(w2v_svc_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",193.547437,1.235733,86.90%,89.51%,89.32%,89.71%,82.53%,82.84%,82.24%,86.02%


# 3. BERT
BERT also turns text into numbers, but it understands sentences much more deeply. Instead of only looking at nearby words, BERT reads the entire sentence at the same time. This lets it understand context, tone, and meaning. For example knowing that “bank” in “I sat by the bank of the river” does not mean a money bank. Because of this, BERT produces much richer and more accurate text representations than Word2Vec.

In summary, Word2Vec knows the relationships between individual words but does not understand the meaning of the whole sentence. In contrast, BERT learns both the sentence and its context, making it far more effective for tasks that require understanding complete ideas.

## 3.1 Generate sample data
In this section, BERT embeddings generated using the `all-MiniLM-L6-v2` model (384 dimensions) are first applied. A stratified 8,000-record subset of the training data is used to efficiently compare different classifier configurations. Once the best-performing configuration is identified, a larger BERT model, `all-mpnet-base-v2` (768 dimensions), is evaluated to determine whether higher-dimensional embeddings provide additional performance gains. Finally, the selected configuration is retrained on the full training dataset and compared with the previous Word2Vec-based classifier results.

In [41]:
X_train_text = X_train.astype(str).tolist()
X_test_text  = X_test.astype(str).tolist()

bert_model_name = 'all-MiniLM-L6-v2'
sbert = SentenceTransformer(bert_model_name)

Xtr_bert = sbert.encode(
    X_train_text,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
Xte_bert = sbert.encode(
    X_test_text,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(Xtr_bert.shape, Xte_bert.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/550 [00:00<?, ?it/s]

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

(70392, 384) (30169, 384)


In [109]:
Xtr_bert

array([[-0.0123732 ,  0.01908213, -0.05738887, ..., -0.03066036,
        -0.05384259, -0.08854051],
       [ 0.00769545, -0.07526441, -0.06705097, ..., -0.04627334,
         0.00426079, -0.03118518],
       [-0.00868034, -0.07438435, -0.07373815, ..., -0.0526348 ,
         0.00389105, -0.11198192],
       ...,
       [ 0.03385962, -0.07651784,  0.09333154, ...,  0.00157036,
        -0.02300527, -0.00139601],
       [-0.0406185 , -0.06525802,  0.03501982, ...,  0.04466414,
         0.00296333, -0.01134   ],
       [-0.03613852, -0.13575359, -0.06689878, ...,  0.02775428,
         0.0742768 , -0.01966565]], shape=(70392, 384), dtype=float32)

In [42]:
# Generate a stratified sample of 8,000 training instances
X_small_bert, _, y_small_bert, _ = train_test_split(
    Xtr_bert,
    y_train,
    train_size=8000,
    stratify=y_train,
    random_state=42
)

## 3.2 Classifier tuning on sampled data
In this section, most classifiers previously evaluated using Word2Vec embeddings are re-examined using BERT representations. XGBClassifier is excluded due to its substantially longer training time and the limited performance gain observed earlier. The parameter search space for each classifier is selected based on the configurations that performed best in the Word2Vec experiments, allowing a more focused and efficient evaluation with BERT embeddings.

In [46]:
# Suppress warnings to improve readability
import os, warnings

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

### 3.2.1 LogisticRegression
Since only 8,000 samples were used for tuning, a slight reduction in performance is expected compared to the full dataset. The best-performing configuration does not lie on the boundary of the searched parameter space, indicating that further expansion of the grid is unlikely to yield substantial improvements. Therefore, the configuration `{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}` is selected as the optimal LogisticRegression setup when using BERT embeddings.

In [49]:
param_grid = {
    'C': [0.5, 1, 2, 5, 10, 20],
    'solver': ['lbfgs'],
    'class_weight': [None, 'balanced'],
    'max_iter': [5000]
}

log_search = random_search_classifier(classifier=LogisticRegression(random_state=42), 
                                      clf_param_grid=param_grid,
                                      X_train=X_small_bert,
                                      y_train=y_small_bert,
                                      n_iter=8)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [50]:
log_search_results = pd.DataFrame(log_search.cv_results_)
show_search_results(log_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%
1,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 20}",0.806292,0.323208,77.08%,82.23%,79.57%,85.06%,67.71%,72.09%,63.84%,74.97%
2,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 1}",0.200395,0.272930,75.86%,81.87%,76.98%,87.43%,63.88%,73.15%,56.71%,72.88%
3,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': 'balanced', 'C': 10}",0.729037,0.312555,75.75%,79.68%,83.42%,76.26%,69.93%,65.58%,74.90%,74.81%
4,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': 'balanced', 'C': 20}",0.414155,0.219411,75.69%,79.63%,83.36%,76.22%,69.85%,65.51%,74.80%,74.74%
5,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': 'balanced', 'C': 2}",0.251180,0.268827,75.51%,79.45%,83.32%,75.92%,69.71%,65.24%,74.83%,74.58%
6,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 0.5}",0.432526,0.369251,75.16%,81.74%,75.46%,89.15%,61.18%,74.33%,51.99%,71.46%
7,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': 'balanced', 'C': 0.5}",0.119586,0.354961,74.55%,78.55%,82.76%,74.76%,68.71%,63.97%,74.20%,73.63%


In [51]:
best_sample_results = pd.DataFrame()
best_log = log_search_results[log_search_results['params'] == {'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}]
best_log.insert(0, 'classifier', 'LogisticRegression')
best_sample_results = pd.concat([best_sample_results, best_log])
show_search_results(best_sample_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LogisticRegression,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%


### 3.2.2 RidgeClassifier
Similarly, the best-performing configuration`{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}`  does not lie on the boundary of the searched parameter space, so it is selected as the optimal RidgeClassifier setup.

In [54]:
param_grid = {
    'alpha': [0.01, 0.05, 0.1, 1.0, 2.0, 5.0],
    'class_weight': [None, 'balanced'],
    'solver': ['auto']
}
ridge_search = random_search_classifier(classifier=RidgeClassifier(random_state=42),
                                        clf_param_grid=param_grid,
                                        X_train=X_small_bert,
                                        y_train=y_small_bert,
                                        n_iter=6)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


In [55]:
ridge_search_results = pd.DataFrame(ridge_search.cv_results_)
show_search_results(ridge_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}",0.157843,0.483245,77.15%,82.29%,79.61%,85.16%,67.79%,72.22%,63.88%,75.04%
1,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.01}",0.149103,0.269397,77.11%,82.25%,79.64%,85.02%,67.80%,72.08%,64.01%,75.02%
2,"{'solver': 'auto', 'class_weight': None, 'alpha': 2.0}",0.134502,0.228356,76.73%,82.33%,78.16%,86.97%,65.91%,73.47%,59.76%,74.12%
3,"{'solver': 'auto', 'class_weight': None, 'alpha': 5.0}",0.185080,0.256847,75.66%,81.81%,76.60%,87.79%,63.23%,73.32%,55.58%,72.52%
4,"{'solver': 'auto', 'class_weight': 'balanced', 'alpha': 2.0}",0.167636,0.319553,75.56%,79.52%,83.28%,76.08%,69.71%,65.35%,74.70%,74.62%
5,"{'solver': 'auto', 'class_weight': 'balanced', 'alpha': 0.1}",0.160484,0.562710,75.36%,79.37%,83.04%,76.00%,69.43%,65.15%,74.30%,74.40%


In [56]:
best_ridge = ridge_search_results[ridge_search_results['params'] == {'solver': 'auto', 'class_weight': None, 'alpha': 0.05}]
best_ridge.insert(0, 'classifier', 'RidgeClassifier')
best_sample_results = pd.concat([best_sample_results, best_ridge])
show_search_results(best_sample_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LogisticRegression,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%
1,RidgeClassifier,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}",0.157843,0.483245,77.15%,82.29%,79.61%,85.16%,67.79%,72.22%,63.88%,75.04%


### 3.2.3 LinearSVC
After exploring different combinations of `C` and `class_weight` hyperparameters, `{'class_weight': None, 'C': 5}` is selected as the best setup for LinearSVC.

In [57]:
param_grid = {
    'C': [0.1, 0.5, 1, 2, 5, 10],
    'class_weight': [None, 'balanced']
}

svc_search = random_search_classifier(classifier=LinearSVC(random_state=42, max_iter=3000),
                                      clf_param_grid=param_grid,
                                      X_train=X_small_bert,
                                      y_train=y_small_bert,
                                      n_iter=8)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [60]:
svc_search_results = pd.DataFrame(svc_search.cv_results_)
show_search_results(svc_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'class_weight': None, 'C': 5}",2.241636,0.488541,77.36%,82.37%,80.05%,84.84%,68.37%,72.14%,64.97%,75.37%
1,"{'class_weight': None, 'C': 10}",2.694255,0.295959,77.26%,82.27%,80.06%,84.60%,68.32%,71.86%,65.11%,75.29%
2,"{'class_weight': None, 'C': 0.5}",1.595254,0.351557,77.08%,82.41%,79.01%,86.11%,67.11%,72.97%,62.12%,74.76%
3,"{'class_weight': 'balanced', 'C': 1}",2.003242,0.359083,75.79%,79.75%,83.32%,76.48%,69.89%,65.71%,74.63%,74.82%
4,"{'class_weight': None, 'C': 0.1}",1.183079,0.427651,75.76%,81.86%,76.75%,87.69%,63.50%,73.32%,56.01%,72.68%
5,"{'class_weight': 'balanced', 'C': 10}",1.781229,0.160776,75.50%,79.49%,83.14%,76.14%,69.58%,65.33%,74.44%,74.54%
6,"{'class_weight': 'balanced', 'C': 5}",2.667227,0.317068,75.49%,79.45%,83.21%,76.02%,69.62%,65.26%,74.60%,74.54%
7,"{'class_weight': 'balanced', 'C': 0.1}",1.154295,0.262463,75.29%,79.23%,83.23%,75.60%,69.50%,64.92%,74.77%,74.36%


In [61]:
best_svc = svc_search_results[svc_search_results['params'] == {'class_weight': None, 'C': 5}]
best_svc.insert(0, 'classifier', 'LinearSVC')
best_sample_results = pd.concat([best_sample_results, best_svc])
show_search_results(best_sample_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'class_weight': None, 'C': 5}",2.241636,0.488541,77.36%,82.37%,80.05%,84.84%,68.37%,72.14%,64.97%,75.37%
1,LogisticRegression,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%
2,RidgeClassifier,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}",0.157843,0.483245,77.15%,82.29%,79.61%,85.16%,67.79%,72.22%,63.88%,75.04%


### 3.2.4 SGDClassifier
`{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.0001}` is selected as the best SGDClassifier setup.

In [76]:
param_grid = {
    'loss': ['hinge', 'log_loss'],
    'alpha': [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1],
    'penalty': ['l2'],
    'class_weight': [None]
}
sgd_search = random_search_classifier(classifier=SGDClassifier(random_state=42, max_iter=1000),
                                      clf_param_grid=param_grid,
                                      X_train=X_small_bert,
                                      y_train=y_small_bert,
                                      n_iter=20)

Fitting 3 folds for each of 14 candidates, totalling 42 fits


In [138]:
show_search_results(sgd_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.0001}",0.593624,0.305398,76.75%,82.30%,78.36%,86.67%,66.13%,73.22%,60.33%,74.21%
1,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 0.0001}",0.891572,0.314869,76.30%,82.41%,76.74%,89.01%,63.66%,75.29%,55.25%,73.03%
2,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 1e-05}",2.013129,0.331784,75.94%,81.62%,78.32%,85.77%,64.40%,72.98%,59.66%,73.01%
3,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 1e-06}",7.712361,0.335111,75.20%,80.43%,79.15%,82.02%,65.90%,68.71%,63.91%,73.16%
4,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.001}",0.533602,0.344734,74.50%,81.58%,74.22%,90.58%,58.53%,75.41%,47.88%,70.06%
5,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 1e-05}",4.145720,0.391041,74.40%,78.06%,83.41%,73.96%,68.88%,64.28%,75.13%,73.47%
6,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 0.001}",0.230746,0.241449,74.08%,81.26%,73.97%,90.16%,57.95%,74.55%,47.44%,69.60%
7,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 1e-06}",6.283891,0.293005,74.06%,78.90%,80.14%,78.41%,65.64%,66.32%,66.87%,72.27%
8,"{'penalty': 'l2', 'loss': 'log_loss', 'class_weight': None, 'alpha': 0.01}",0.187105,0.210188,63.04%,77.11%,62.81%,99.86%,3.99%,91.00%,2.06%,40.55%
9,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.01}",0.130826,0.249745,62.35%,76.81%,62.35%,100.00%,0.00%,0.00%,0.00%,38.40%


In [68]:
best_sgd = sgd_search_results[sgd_search_results['params'] == {'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.0001}]
best_sgd.insert(0, 'classifier', 'SGDClassifier')
best_sample_results = pd.concat([best_sample_results, best_sgd])
show_search_results(best_sample_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'class_weight': None, 'C': 5}",2.241636,0.488541,77.36%,82.37%,80.05%,84.84%,68.37%,72.14%,64.97%,75.37%
1,LogisticRegression,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%
2,RidgeClassifier,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}",0.157843,0.483245,77.15%,82.29%,79.61%,85.16%,67.79%,72.22%,63.88%,75.04%
3,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.0001}",0.620170,0.377596,76.75%,82.30%,78.36%,86.67%,66.13%,73.22%,60.33%,74.21%


### 3.2.5 HistGradientBoosting
After two rounds of hyperparameter tuning with an expanded search range, the configuration `{'max_leaf_nodes': 31, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 10.0}` was identified as the best-performing setup for the HistGradientBoostingClassifier. Its performance on the 8,000-record sample is lower than that of the other classifiers, likely because boosting models typically require larger training sets to fully benefit from their higher model capacity. With a limited sample size, the model may underfit and fail to learn stable tree splits. Therefore, despite its weaker performance on the sample data, HistGradientBoosting remains a candidate worth retraining on the full dataset, where it may achieve substantially better results.

In [137]:
param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_iter': [500, 800],
    'max_depth': [10, None],
    'max_leaf_nodes': [63, 127, 255],
    'l2_regularization': [1.0, 3.0, 5.0]
}

hgb_search = random_search_classifier(
    classifier=HistGradientBoostingClassifier(random_state=42, early_stopping=True),
    clf_param_grid=param_grid,
    X_train=X_small_bert,
    y_train=y_small_bert,
    n_iter=18
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


In [73]:
hgb_search_results = pd.DataFrame(hgb_search.cv_results_)
show_search_results(hgb_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 5.0}",59.650821,1.146339,72.08%,79.83%,72.61%,88.65%,54.61%,70.36%,44.62%,67.22%
1,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.1, 'l2_regularization': 3.0}",45.441050,0.997629,71.86%,79.57%,72.70%,87.87%,54.83%,69.33%,45.35%,67.20%
2,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.1, 'l2_regularization': 1.0}",38.613375,0.754112,71.84%,79.59%,72.59%,88.09%,54.56%,69.48%,44.92%,67.08%
3,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 1.0}",26.218229,0.768526,71.84%,79.62%,72.54%,88.25%,54.41%,69.70%,44.65%,67.02%
4,"{'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}",58.400751,2.077798,71.79%,79.74%,72.20%,89.03%,53.57%,70.41%,43.23%,66.65%
5,"{'max_leaf_nodes': 127, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 3.0}",89.929410,0.937639,71.56%,79.76%,71.69%,89.88%,52.19%,71.12%,41.24%,65.98%
6,"{'max_leaf_nodes': 127, 'max_iter': 800, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 3.0}",76.508733,1.054791,71.56%,79.76%,71.69%,89.88%,52.19%,71.12%,41.24%,65.98%
7,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 1.0}",42.730992,0.820997,71.54%,79.56%,72.04%,88.83%,53.15%,69.89%,42.90%,66.35%
8,"{'max_leaf_nodes': 255, 'max_iter': 500, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 5.0}",73.060585,1.143658,71.50%,79.66%,71.77%,89.49%,52.41%,70.57%,41.70%,66.04%
9,"{'max_leaf_nodes': 127, 'max_iter': 800, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 1.0}",62.087423,0.905120,71.47%,79.72%,71.60%,89.94%,51.90%,71.08%,40.90%,65.81%


In [75]:
param_grid = {
    'learning_rate': [0.01, 0.05],
    'max_iter': [300, 500],
    'max_depth': [None],
    'max_leaf_nodes': [31, 63],
    'l2_regularization': [5.0, 10.0]
}

hgb_search = random_search_classifier(
    classifier=HistGradientBoostingClassifier(random_state=42, early_stopping=True),
    clf_param_grid=param_grid,
    X_train=X_small_bert,
    y_train=y_small_bert,
    n_iter=6
)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


In [78]:
hgb_search_results = pd.DataFrame(hgb_search.cv_results_)
show_search_results(hgb_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'max_leaf_nodes': 31, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 10.0}",46.434566,0.802226,73.03%,80.38%,73.55%,88.63%,56.81%,71.48%,47.18%,68.60%
1,"{'max_leaf_nodes': 63, 'max_iter': 300, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 10.0}",68.831843,2.277110,72.61%,80.22%,72.98%,89.05%,55.51%,71.47%,45.39%,67.86%
2,"{'max_leaf_nodes': 63, 'max_iter': 300, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 5.0}",57.342987,0.925344,72.08%,79.83%,72.61%,88.65%,54.61%,70.36%,44.62%,67.22%
3,"{'max_leaf_nodes': 63, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.01, 'l2_regularization': 10.0}",128.054534,0.972712,71.65%,79.88%,71.65%,90.24%,52.05%,71.65%,40.87%,65.96%
4,"{'max_leaf_nodes': 63, 'max_iter': 300, 'max_depth': None, 'learning_rate': 0.01, 'l2_regularization': 5.0}",108.493133,1.870410,70.53%,79.42%,70.32%,91.22%,48.08%,71.38%,36.25%,63.75%
5,"{'max_leaf_nodes': 31, 'max_iter': 300, 'max_depth': None, 'learning_rate': 0.01, 'l2_regularization': 5.0}",63.158443,1.938068,70.35%,79.33%,70.17%,91.24%,47.58%,71.14%,35.76%,63.45%


In [79]:
best_hgb = hgb_search_results[hgb_search_results['params'] == {'max_leaf_nodes': 31, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 10.0}]
best_hgb.insert(0, 'classifier', 'HistGradientBoostingClassifier')
best_sample_results = pd.concat([best_sample_results, best_hgb])
show_search_results(best_sample_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'class_weight': None, 'C': 5}",2.241636,0.488541,77.36%,82.37%,80.05%,84.84%,68.37%,72.14%,64.97%,75.37%
1,LogisticRegression,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%
2,RidgeClassifier,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}",0.157843,0.483245,77.15%,82.29%,79.61%,85.16%,67.79%,72.22%,63.88%,75.04%
3,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.0001}",0.620170,0.377596,76.75%,82.30%,78.36%,86.67%,66.13%,73.22%,60.33%,74.21%
4,HistGradientBoostingClassifier,"{'max_leaf_nodes': 31, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 10.0}",46.434566,0.802226,73.03%,80.38%,73.55%,88.63%,56.81%,71.48%,47.18%,68.60%


### 3.2.6 LightGBM
After two rounds of hyperparameter tuning with an expanded search range, the configuration `{'subsample': 0.5, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}` was identified as the best-performing setup for LightGBM. Similar to HistGradientBoosting, LightGBM is also a boosting model and generally benefits from larger training sets to fully exploit its model capacity. As a result, its performance on the 8,000-record sample is weaker compared with the linear classifiers, likely due to underfitting on the limited data. Despite the lower performance in the sample-based evaluation, LightGBM remains a reasonable candidate to retrain on the full dataset, where its performance may improve significantly.

In [80]:
param_grid = {
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 63],
    'n_estimators': [300, 500],
    'feature_fraction': [0.7, 0.9],
    'subsample': [0.7, 0.9]
}

lgbm_search = random_search_classifier(
    classifier=LGBMClassifier(
        random_state=42,
        objective='binary',
        n_jobs=-1
    ),
    clf_param_grid=param_grid,
    X_train=X_small_bert,
    y_train=y_small_bert,       
    n_iter=12
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Info] Number of positive: 3012, number of negative: 4988
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008927 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 384
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376500 -> initscore=-0.504431
[LightGBM] [Info] Start training from score -0.504431


In [82]:
lgbm_search_results = pd.DataFrame(lgbm_search.cv_results_)
show_search_results(lgbm_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}",46.523980,1.108677,73.83%,80.67%,74.74%,87.63%,59.45%,71.36%,50.96%,70.06%
1,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.1, 'feature_fraction': 0.7}",29.464795,1.199499,73.40%,80.38%,74.40%,87.43%,58.68%,70.75%,50.17%,69.53%
2,"{'subsample': 0.9, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.1, 'feature_fraction': 0.7}",31.451940,0.952452,73.40%,80.38%,74.40%,87.43%,58.68%,70.75%,50.17%,69.53%
3,"{'subsample': 0.9, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.05, 'feature_fraction': 0.9}",39.479832,1.233124,73.10%,80.31%,73.86%,87.99%,57.55%,70.89%,48.44%,68.93%
4,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.05, 'feature_fraction': 0.9}",27.347173,0.324058,73.10%,80.31%,73.86%,87.99%,57.55%,70.89%,48.44%,68.93%
5,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.1, 'feature_fraction': 0.9}",43.202286,2.453033,73.09%,80.08%,74.35%,86.77%,58.52%,69.71%,50.43%,69.30%
6,"{'subsample': 0.9, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.1, 'feature_fraction': 0.9}",31.672609,1.174097,73.09%,80.08%,74.35%,86.77%,58.52%,69.71%,50.43%,69.30%
7,"{'subsample': 0.9, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.9}",72.287552,0.909772,73.00%,80.02%,74.28%,86.73%,58.37%,69.58%,50.27%,69.19%
8,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.05, 'feature_fraction': 0.7}",32.154226,0.884232,72.99%,80.18%,73.90%,87.63%,57.60%,70.41%,48.74%,68.89%
9,"{'subsample': 0.7, 'num_leaves': 63, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.9}",96.194623,0.785508,72.99%,80.22%,73.80%,87.87%,57.40%,70.66%,48.34%,68.81%


In [84]:
param_grid = {
    'learning_rate': [0.1],
    'num_leaves': [31],
    'n_estimators': [500, 800],
    'feature_fraction': [0.5, 0.7],
    'subsample': [0.5, 0.7]
}

lgbm_search = random_search_classifier(
    classifier=LGBMClassifier(
        random_state=42,
        objective='binary',
        n_jobs=-1
    ),
    clf_param_grid=param_grid,
    X_train=X_small_bert,
    y_train=y_small_bert,       
    n_iter=8
)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Info] Number of positive: 3012, number of negative: 4988
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009231 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 384
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376500 -> initscore=-0.504431
[LightGBM] [Info] Start training from score -0.504431


In [87]:
lgbm_search_results = pd.DataFrame(lgbm_search.cv_results_)
show_search_results(lgbm_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'subsample': 0.5, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}",71.766000,1.602700,73.83%,80.67%,74.74%,87.63%,59.45%,71.36%,50.96%,70.06%
1,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}",70.023556,1.634556,73.83%,80.67%,74.74%,87.63%,59.45%,71.36%,50.96%,70.06%
2,"{'subsample': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'learning_rate': 0.1, 'feature_fraction': 0.5}",109.817174,2.142846,73.66%,80.51%,74.75%,87.23%,59.41%,70.77%,51.20%,69.96%
3,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 800, 'learning_rate': 0.1, 'feature_fraction': 0.5}",97.583943,1.733347,73.66%,80.51%,74.75%,87.23%,59.41%,70.77%,51.20%,69.96%
4,"{'subsample': 0.5, 'num_leaves': 31, 'n_estimators': 800, 'learning_rate': 0.1, 'feature_fraction': 0.7}",96.058190,0.723133,73.64%,80.53%,74.62%,87.47%,59.17%,70.99%,50.73%,69.85%
5,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 800, 'learning_rate': 0.1, 'feature_fraction': 0.7}",79.127839,0.306940,73.64%,80.53%,74.62%,87.47%,59.17%,70.99%,50.73%,69.85%
6,"{'subsample': 0.5, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.5}",55.563745,1.291303,73.39%,80.27%,74.64%,86.81%,59.14%,70.08%,51.16%,69.71%
7,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.5}",55.266132,1.311339,73.39%,80.27%,74.64%,86.81%,59.14%,70.08%,51.16%,69.71%


In [88]:
best_lgbm = lgbm_search_results[lgbm_search_results['params'] == {'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}]
best_lgbm.insert(0, 'classifier', 'LGBMClassifier')
best_sample_results = pd.concat([best_sample_results, best_lgbm])
show_search_results(best_sample_results)

,classifier,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,LinearSVC,"{'class_weight': None, 'C': 5}",2.241636,0.488541,77.36%,82.37%,80.05%,84.84%,68.37%,72.14%,64.97%,75.37%
1,LogisticRegression,"{'solver': 'lbfgs', 'max_iter': 5000, 'class_weight': None, 'C': 10}",0.407984,0.234227,77.33%,82.46%,79.64%,85.49%,67.94%,72.64%,63.81%,75.20%
2,RidgeClassifier,"{'solver': 'auto', 'class_weight': None, 'alpha': 0.05}",0.157843,0.483245,77.15%,82.29%,79.61%,85.16%,67.79%,72.22%,63.88%,75.04%
3,SGDClassifier,"{'penalty': 'l2', 'loss': 'hinge', 'class_weight': None, 'alpha': 0.0001}",0.620170,0.377596,76.75%,82.30%,78.36%,86.67%,66.13%,73.22%,60.33%,74.21%
4,LGBMClassifier,"{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}",70.023556,1.634556,73.83%,80.67%,74.74%,87.63%,59.45%,71.36%,50.96%,70.06%
5,HistGradientBoostingClassifier,"{'max_leaf_nodes': 31, 'max_iter': 500, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 10.0}",46.434566,0.802226,73.03%,80.38%,73.55%,88.63%,56.81%,71.48%,47.18%,68.60%


In [ ]:
best_sample_results.to_csv('../data/interim/best_sample_results.csv', index=False)

## 3.3 Final training
In this section, the best configurations identified in the previous experiments are retrained on the full training dataset. The strongest classifier will then be applied to the second BERT model to examine whether the performance can be further improved.

### 3.3.1 Using full dataset
LinearSVC, LGBMClassifier, and HistGradientBoostingClassifier are selected for evaluation on the full training dataset. LinearSVC is included because it achieved the best overall performance on the 8,000-sample subset, while LGBMClassifier and HistGradientBoostingClassifier are boosting-based models that may benefit from a larger dataset and potentially perform better when trained on the full data.

The results show that LinearSVC remains the best-performing model. Therefore, it is used in the next stage to train on the alternative BERT representation.

In [102]:
param_grid = {}

svc_search = random_search_classifier(
    classifier=LinearSVC(
        random_state=42, 
        max_iter=3000, 
        class_weight=None, 
        C=5
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_bert,
    y_train=y_train,       
    n_iter=1
)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [140]:
lgbm_search = random_search_classifier(
    classifier=LGBMClassifier(
        random_state=42,
        objective='binary',
        n_jobs=-1,
        subsample=0.7, 
        num_leaves=31, 
        n_estimators=500, 
        learning_rate=0.1, 
        feature_fraction=0.7
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_bert,
    y_train=y_train,       
    n_iter=1
)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [139]:
hgb_search = random_search_classifier(
    classifier=HistGradientBoostingClassifier(
        random_state=42, 
        early_stopping=True,
        max_leaf_nodes=31, 
        max_iter=500, 
        max_depth=None, 
        learning_rate=0.05, 
        l2_regularization=10.0
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_bert,
    y_train=y_train,       
    n_iter=1
)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [108]:
print("svc_search")
show_search_results(pd.DataFrame(svc_search.cv_results_))
print("lgbm_search")
show_search_results(pd.DataFrame(lgbm_search.cv_results_))
print("hgb_search")
show_search_results(pd.DataFrame(hgb_search.cv_results_))

svc_search


,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,{},14.577522,1.041372,78.61%,83.46%,80.57%,86.57%,69.72%,74.63%,65.41%,76.59%


lgbm_search


,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,{},76.210996,1.793265,76.29%,82.01%,77.80%,86.70%,65.21%,72.83%,59.04%,73.61%


hgb_search


,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,{},123.130051,5.423129,76.42%,82.22%,77.59%,87.45%,65.00%,73.67%,58.16%,73.61%


### 3.3.2 Using a different BERT model

To examine whether a higher-dimensional embedding could improve performance, the `all-mpnet-base-v2` model (768-dimensional) was used with the best-performing classifier, LinearSVC. The results show a noticeable improvement compared with `all-MiniLM-L6-v2`. However, generating mpnet embeddings required significantly more computation time, making the overall pipeline considerably slower. Despite the improvement, the final performance still fell short of the best Word2Vec-based configuration.

In [110]:
bert_model_name = 'all-mpnet-base-v2'
sbert = SentenceTransformer(bert_model_name)

Xtr_bert_v2 = sbert.encode(
    X_train_text,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
Xte_bert_v2 = sbert.encode(
    X_test_text,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(Xtr_bert_v2.shape, Xte_bert_v2.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/550 [00:00<?, ?it/s]

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

(70392, 768) (30169, 768)


In [111]:
param_grid = {}

svc_search = random_search_classifier(
    classifier=LinearSVC(
        random_state=42, 
        max_iter=3000, 
        class_weight=None, 
        C=5
    ),
    clf_param_grid=param_grid,
    X_train=Xtr_bert_v2,
    y_train=y_train,       
    n_iter=1
)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [112]:
print("svc_search")
show_search_results(pd.DataFrame(svc_search.cv_results_))

svc_search


,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,{},57.209165,1.080645,81.32%,85.45%,83.09%,87.94%,73.93%,77.89%,70.36%,79.69%


[CV] END .................................................... total time=  57.9s
[CV] END .................................................... total time=  58.5s
[CV] END .................................................... total time=  58.5s


# 4. Final model selection
After evaluating various embedding methods and classifier configurations, the final selected model consists of:
- Embedding: Word2Vec with configuration `{'vector_size': 1000, 'min_count': 10, 'sg': 1, 'epochs': 10}`
- Classifier: LinearSVC with configuration `{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}`

This combination achieved the best overall performance, offering strong generalization ability and consistent results across both cross-validation and the held-out test set.

Final Test Performance:
- Accuracy: 87.41%
- AI F1: 89.95%	
- AI Precision: 89.49%	
- AI Recall: 90.43%	
- Human F1: 83.13%	
- Human Precision: 83.86%
- Human Recall: 82.41%	
- Macro F1: 86.54%


Although the selected Word2Vec + LinearSVC model performs the best among the embedding-based approaches tested in Assignment 2, its performance is still slightly worse than the best model from Assignment 1, where `LogisticRegression(max_iter=1000, solver='saga')` combined with `CountVectorizer` achieved higher accuracy and macro-F1. This difference can be explained by the characteristics of the feature representations:
- CountVectorizer preserves fine-grained word-level frequency information across a very high-dimensional sparse space (22,912 features in Assignment 1). Since the task of distinguishing AI-generated text from human-written text is largely driven by stylistic and lexical differences (e.g., preferred wording, repetitiveness, formality, phrasing patterns), sparse lexical features tend to capture these signals more effectively.
- In contrast, Word2Vec compresses each sentence into a dense 1000-dimensional representation by averaging word embeddings. This process smooths out many discriminative lexical details that CountVectorizer captures, making the resulting features less sensitive to subtle stylistic cues. Similarly, BERT-based sentence embeddings focus on semantic meaning rather than lexical style, which further reduces their effectiveness for this style-based classification task. 

Therefore, it is reasonable that the Assignment 1 model using sparse lexical features outperforms the embedding-based models in Assignment 2, despite the latter being more modern representation methods.

In [128]:
w2v = {
    'vector_size': 1000,
    'min_count': 10,
    'sg': 1,
    'epochs': 10
}

fixed_w2v = Word2Vec(
    sentences=tokenized_train,
    seed=42,
    workers=1,
    **w2v
)

Xtr = vectorize_texts(fixed_w2v, tokenized_train)
Xts = vectorize_texts(fixed_w2v, tokenized_test)

scaler = StandardScaler(with_mean=True, with_std=True)
Xtr_s  = scaler.fit_transform(Xtr)
Xts_s  = scaler.transform(Xts)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [129]:
svc = LinearSVC(random_state=42, max_iter=3000, loss='hinge', dual=True, class_weight=None, C=0.1)
svc.fit(Xtr_s, y_train)
y_pred = svc.predict(Xts_s)

In [130]:
metrics = {
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    
    "ai_f1": round(f1_score(y_test, y_pred, pos_label='AI', zero_division=0), 4),
    "ai_precision": round(precision_score(y_test, y_pred, pos_label='AI', zero_division=0), 4),
    "ai_recall": round(recall_score(y_test, y_pred, pos_label='AI', zero_division=0), 4),
    
    "human_f1": round(f1_score(y_test, y_pred, pos_label='Human', zero_division=0), 4),
    "human_precision": round(precision_score(y_test, y_pred, pos_label='Human', zero_division=0), 4),
    "human_recall": round(recall_score(y_test, y_pred, pos_label='Human', zero_division=0), 4),
    
    "macro_f1": round(f1_score(y_test, y_pred, average='macro'), 4)
}
# Performance of the selected model on the testing data
results_df = pd.DataFrame([metrics])

show_styled_df(results_df)

,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,87.41%,89.95%,89.49%,90.43%,83.13%,83.86%,82.41%,86.54%


In [95]:
# Performance of 3-fold cross-validation on the training data
show_search_results(w2v_svc_search_results)

,params,mean_fit_time,mean_score_time,accuracy,ai_f1,ai_precision,ai_recall,human_f1,human_precision,human_recall,macro_f1
0,"{'loss': 'hinge', 'dual': True, 'class_weight': None, 'C': 0.1}",193.547437,1.235733,86.90%,89.51%,89.32%,89.71%,82.53%,82.84%,82.24%,86.02%


# 5. Classify new data
According to the selected model, the predicted results are as follows:

- Test A: Huamn 0.685, AI 0.315
- Test B: Human 0.91,  AI 0.09

These results indicate that Test A contains a substantially higher proportion of text classified as AI-generated compared with Test B. While the absolute percentages differ from those reported in Assignment 1, the overall conclusion is consistent: Test A appears to contain more AI-generated content than Test B.

It is important to note that these values represent model-based estimates rather than exact proportions. The classifier may mislabel individual instances, and embeddings can smooth out stylistic differences. Therefore, the results allow us to compare the two datasets relatively, but do not provide a definitive measurement of the true percentage of AI-generated text.

In [ ]:
test_a = pd.read_csv('../data/raw/test_Abst_A.csv')
test_b = pd.read_csv('../data/raw/test_Abst_B.csv')
display(test_a.shape)
display(test_b.shape)

(1000, 1)

(1000, 1)

In [131]:
tokenized_a = [gensim_tokenize(s) for s in test_a['clean_abstract']]
tokenized_b = [gensim_tokenize(s) for s in test_b['clean_abstract']]

In [132]:
vec_a  = vectorize_texts(fixed_w2v, tokenized_a)
vec_b  = vectorize_texts(fixed_w2v, tokenized_b)

vec_a_s = scaler.transform(vec_a)
vec_b_s = scaler.transform(vec_b)

pred_a = svc.predict(vec_a_s)
pred_b = svc.predict(vec_b_s)

In [134]:
a_result = pd.concat([test_a, pd.DataFrame(pred_a, columns=['predicted_label'])], axis=1)
b_result = pd.concat([test_b, pd.DataFrame(pred_b, columns=['predicted_label'])], axis=1)

print("Test A:")
display(a_result['predicted_label'].value_counts(normalize=True))

print("Test B:")
display(b_result['predicted_label'].value_counts(normalize=True))

Test A:


predicted_label
Human    0.685
AI       0.315
Name: proportion, dtype: float64

Test B:


predicted_label
Human    0.91
AI       0.09
Name: proportion, dtype: float64